# FlyRec: a connectome-inspired alternative to SASRec

**Status: prototype; syntax checked, not executed or trained in the authoring environment.**
Use a GPU Colab runtime. This notebook embeds your earlier SparseWalker benchmark,
reuses its SASRec, data splits and full-catalog evaluator, and adds a recurrent graph encoder.
No FlyGM performance result is claimed. No RL or distillation is used.

Each item updates a persistent graph state; a learned readout predicts the next item.
The experimental hypothesis is whether biological connectivity transfers to recommendation.

The pilot uses a **512-neuron induced subgraph**, selected by total degree, from a real
FlyWire connections CSV. This selection biases the graph toward hubs and removes external
connections. It does not reproduce the whole-brain model. Inputs and outputs are learned
over all retained nodes, rather than anatomically assigned afferent/efferent populations.
We use unweighted edges, shared gated updates and eight state channels per node.

Required input: download a connections CSV (or CSV.gz) from
[FlyWire Codex](https://codex.flywire.ai), then upload it when prompted.
Expected columns: `pre_root_id`, `post_root_id`; optional `syn_count`.
Record the dataset release in the configuration. The notebook never silently substitutes
a random graph for biological data. Random graphs below are explicitly labeled smoke fixtures.

Sources: [FlyGM](https://lnsgroup.cc/research/FlyGM/),
[paper](https://arxiv.org/abs/2602.17997), [FlyWire](https://flywire.ai).


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, time
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
print(torch.__version__, torch.cuda.get_device_name())
RUN_DIR = Path('/content/flyrec_runs') / time.strftime('%Y%m%d_%H%M%S')
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ['SW_ONLY_MODEL']='SASRec'
os.environ['SW_ONLY_DATASET']='beauty'
os.environ['SPARSEWALKER_ROOT']=str(RUN_DIR)
os.environ['SPARSEWALKER_DATA']='/content/flyrec_data'


The embedded benchmark is a copy; your original file is unchanged.


In [ ]:
BENCHMARK_SOURCE = '# ============================================================\n# SparseWalker paper benchmark — standalone Colab/Python script\n# ============================================================\n# Primary protocol: temporal leave-two-out, full-catalog ranking,\n# seen-item masking, ID-only input. Secondary: deterministic 100-neg LOO.\n#\n# Models:\n#   MostPop, GRU4Rec, SASRec, SASRec+SS, eSASRec,\n#   HSTU-core, HSTU-large, BERT4Rec, FMLP-Rec,\n#   Mamba4Rec, PersistentSparseWalker.\n#\n# Datasets:\n#   Amazon Beauty, Video Games, Sports & Outdoors, Toys & Games,\n#   MovieLens-1M, MovieLens-20M; optional Amazon Books.\n#\n# Notes for paper use:\n# - HSTU-core implements the public HSTU pointwise-aggregated-attention\n#   block (U,V,Q,K + SiLU + LN(A V) * U), without Meta\'s production\n#   temporal RAB / stochastic-length / custom fused HSTU kernel. It is\n#   intentionally labeled HSTU-core in result tables.\n# - Mamba4Rec uses mamba_ssm.Mamba and mirrors the official Mamba4Rec\n#   block wrapper (Mamba -> dropout/residual/LN -> FFN).\n# - eSASRec is SASRec objective + LiGR-style gated pre-LN blocks +\n#   shared sampled-softmax. SASRec+SS isolates the loss contribution.\n# - FMLP-Rec uses learnable complex frequency filters + FFN blocks.\n# - BERT4Rec uses a Cloze/masked-item objective and MASK-at-end eval.\n# - All main learned models use item IDs only.\n# ============================================================\n\n# %% [markdown]\n# 0. Optional Colab dependency install\n\n# In Colab run this cell once before importing Mamba4Rec if mamba_ssm is absent:\n# !pip -q install ninja einops packaging\n# !pip -q install "mamba-ssm[causal-conv1d]" --no-build-isolation\n\n# %%\nimport os, sys, ast, gc, gzip, json, math, random, shutil, time, urllib.request, zipfile\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass, asdict, field\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple, Optional, Any\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\n\ntry:\n    import matplotlib.pyplot as plt\nexcept Exception:\n    plt = None\n\ntry:\n    import triton\n    import triton.language as tl\n    HAS_TRITON = True\nexcept Exception:\n    HAS_TRITON = False\n\ntry:\n    from mamba_ssm import Mamba\n    HAS_MAMBA = True\n    MAMBA_IMPORT_ERROR = None\nexcept Exception as e:\n    Mamba = None\n    HAS_MAMBA = False\n    MAMBA_IMPORT_ERROR = repr(e)\n\n# %%\n# ============================================================\n# 1. CONFIG\n# ============================================================\n\n@dataclass\nclass DatasetSpec:\n    name: str\n    kind: str                    # amazon | ml1m | ml20m\n    url: str\n    filename: str\n    archive_member: Optional[str] = None\n    default_max_len: int = 50\n    large_catalog: bool = False\n\nDATASETS: Dict[str, DatasetSpec] = {\n    "beauty": DatasetSpec(\n        "beauty", "amazon",\n        "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Beauty_5.json.gz",\n        "reviews_Beauty_5.json.gz", default_max_len=50),\n    "video_games": DatasetSpec(\n        "video_games", "amazon",\n        "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Video_Games_5.json.gz",\n        "reviews_Video_Games_5.json.gz", default_max_len=50),\n    "sports": DatasetSpec(\n        "sports", "amazon",\n        "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz",\n        "reviews_Sports_and_Outdoors_5.json.gz", default_max_len=50),\n    "toys": DatasetSpec(\n        "toys", "amazon",\n        "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Toys_and_Games_5.json.gz",\n        "reviews_Toys_and_Games_5.json.gz", default_max_len=50),\n    "ml1m": DatasetSpec(\n        "ml1m", "ml1m",\n        "https://files.grouplens.org/datasets/movielens/ml-1m.zip",\n        "ml-1m.zip", archive_member="ml-1m/ratings.dat", default_max_len=200),\n    "ml20m": DatasetSpec(\n        "ml20m", "ml20m",\n        "https://files.grouplens.org/datasets/movielens/ml-20m.zip",\n        "ml-20m.zip", archive_member="ml-20m/ratings.csv", default_max_len=200),\n    "books": DatasetSpec(\n        "books", "amazon",\n        "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Books_5.json.gz",\n        "reviews_Books_5.json.gz", default_max_len=50, large_catalog=True),\n}\n\n@dataclass\nclass Config:\n    # ------------------------------------------------------------\n    # FAST DEBUG SELECTORS\n    # Set any of these to None to restore the normal multi-run suite.\n    # For HSTU debugging, change only_model between:\n    #   "HSTU-core" and "HSTU-large"\n    # ------------------------------------------------------------\n    only_model: Optional[str] = os.environ.get("SW_ONLY_MODEL", "HSTU-core") or None\n    only_dataset: Optional[str] = os.environ.get("SW_ONLY_DATASET", "beauty") or None\n    only_seed: Optional[int] = int(os.environ.get("SW_ONLY_SEED", "42")) if os.environ.get("SW_ONLY_SEED", "42") else None\n\n    # Run profile\n    profile: str = "paper_full"  # smoke | main | paper_full\n    datasets: Tuple[str, ...] = ("beauty", "video_games", "sports", "toys", "ml1m", "ml20m")\n    include_books: bool = False\n    seeds: Tuple[int, ...] = (42, 43, 44)\n\n    # Models. HSTU-large is expensive and is run in paper_full only by default.\n    models: Tuple[str, ...] = (\n        "MostPop", "GRU4Rec", "SASRec", "SASRec+SS", "eSASRec",\n        "HSTU-core", "HSTU-large", "BERT4Rec", "FMLP-Rec",\n        "Mamba4Rec", "SparseWalker", "SparseWalker-E2E",\n    )\n\n    # Paths\n    root: str = os.environ.get("SPARSEWALKER_ROOT", "/content/sparsewalker_paper")\n    data_dir: str = os.environ.get("SPARSEWALKER_DATA", "/content/sparsewalker_data")\n    reuse: bool = True\n    save_predictions: bool = False\n\n    # Protocol\n    min_user_len: int = 5\n    max_len_override: Optional[int] = None\n    topks: Tuple[int, ...] = (10, 20, 50, 200)\n    sampled_eval_negs: int = 100\n    head_fraction: float = 0.20\n\n    # Shared architecture for matched-capacity comparisons\n    d_model: int = 64\n    layers: int = 2\n    heads: int = 2\n    ffn_dim: int = 256\n    dropout: float = 0.20\n\n    # Training\n    batch_size: int = 512\n    eval_batch_size: int = 1024\n    max_epochs: int = 30\n    patience: int = 6\n    weight_decay: float = 1e-4\n    grad_clip: float = 5.0\n    warmup_epochs: int = 3\n    min_lr: float = 1e-4\n    num_workers: int = 0\n\n    # Per-model peak LR; chosen as reasonable starting recipes, not hidden tuning.\n    lr_by_model: Dict[str, float] = field(default_factory=lambda: {\n        "GRU4Rec": 1e-3,\n        "SASRec": 5e-3,\n        "SASRec+SS": 2e-3,\n        "eSASRec": 2e-3,\n        "HSTU-core": 1e-3,\n        "HSTU-large": 1e-3,\n        "BERT4Rec": 1e-3,\n        "FMLP-Rec": 1e-3,\n        "Mamba4Rec": 1e-3,\n        "SparseWalker": 1e-3,\n        "SparseWalker-E2E": 1e-3,\n    })\n    n_negs_train: int = 256\n\n    # BERT4Rec\n    bert_mask_prob: float = 0.20\n\n    # Mamba4Rec official block knobs\n    mamba_layers: int = 1          # official Mamba4Rec default; keep separate from shared 2-layer baselines\n    mamba_d_state: int = 32\n    mamba_d_conv: int = 4\n    mamba_expand: int = 2\n\n    # HSTU\n    # "paper" uses Meta\'s public training recipe as closely as possible\n    # while keeping this benchmark ID-only (position bias, no timestamp feature).\n    hstu_recipe: str = "paper"\n    hstu_temperature: float = 0.05\n    hstu_l2_eps: float = 1e-6\n    hstu_amazon_negatives: int = 512\n    hstu_movielens_negatives: int = 128\n    hstu_batch_size: int = 128\n    hstu_max_epochs: int = 101\n    hstu_amazon_max_epochs: int = 80   # debug-friendly cap; raise to 201 for paper-faithful Amazon/Books\n    hstu_patience: int = 20\n    hstu_large_layers: int = 16\n    hstu_large_heads: int = 8\n\n    # Walker concepts\n    concept_side: int = 256\n    concept_dim: int = 16\n    active_concepts: int = 8\n    fresh_top_side: int = 2\n    graph_degree: int = 4\n    fresh_weight: float = 0.25\n    pursuit_start: int = 4\n    pursuit_every: int = 2\n    pursuit_refresh: int = 2\n\n    # Walker terminal compiler\n    teacher_topk: int = 32\n    compiler_teacher_k: int = 32\n    compiler_max_events: int = 200_000\n    compiler_chunk_events: int = 5_000\n    target_bonus: float = 4.0\n    terminal_max_degree: int = 512\n    degree_sweep: Tuple[int, ...] = (64, 128, 256, 512)\n\n    # End-to-end sparse terminal learning\n    e2e_terminal_degree: int = 128\n    e2e_loss_chunk: int = 256\n    e2e_evidence_per_batch: int = 2048\n    e2e_terminal_refresh_start: int = 1\n    e2e_terminal_refresh_every: int = 1\n\n    # Triton serving\n    score_block: int = 128\n    block_keep: int = 16\n\n    # Benchmarks\n    run_latency: bool = True\n    latency_repeats: int = 200\n    latency_warmup: int = 30\n\n    # Optional long-context quality ablation on ML20M.\n    # It retrains SASRec/HSTU/Mamba/Walker at each length and can be expensive.\n    run_long_context_ablation: bool = False\n    long_context_lengths: Tuple[int, ...] = (50, 128, 256, 512)\n    long_context_models: Tuple[str, ...] = ("SASRec", "HSTU-core", "Mamba4Rec", "SparseWalker", "SparseWalker-E2E")\n    long_context_seed: int = 42\n\nCFG = Config()\n\nif CFG.profile == "smoke":\n    CFG.datasets = ("beauty",)\n    CFG.seeds = (42,)\n    CFG.models = ("SASRec", "HSTU-core", "Mamba4Rec", "SparseWalker", "SparseWalker-E2E")\n    CFG.max_epochs = 2\n    CFG.patience = 2\n    CFG.compiler_max_events = 20_000\n    CFG.degree_sweep = (64, 128)\nelif CFG.profile == "main":\n    CFG.datasets = ("beauty", "video_games", "sports", "toys", "ml1m")\n    CFG.seeds = (42,)\n    CFG.models = tuple(m for m in CFG.models if m != "HSTU-large")\n\nif CFG.include_books and "books" not in CFG.datasets:\n    CFG.datasets = tuple(CFG.datasets) + ("books",)\n\n# Apply debug selectors last so they override profile defaults.\nif CFG.only_dataset is not None:\n    if CFG.only_dataset not in DATASETS:\n        raise ValueError(f"Unknown only_dataset={CFG.only_dataset!r}")\n    CFG.datasets = (CFG.only_dataset,)\nif CFG.only_seed is not None:\n    CFG.seeds = (int(CFG.only_seed),)\nif CFG.only_model is not None:\n    valid_models = {\n        "MostPop","GRU4Rec","SASRec","SASRec+SS","eSASRec",\n        "HSTU-core","HSTU-large","BERT4Rec","FMLP-Rec",\n        "Mamba4Rec","SparseWalker","SparseWalker-E2E",\n    }\n    if CFG.only_model not in valid_models:\n        raise ValueError(f"Unknown only_model={CFG.only_model!r}")\n    CFG.models = (CFG.only_model,)\n\nROOT = Path(CFG.root); ROOT.mkdir(parents=True, exist_ok=True)\nDATA_ROOT = Path(CFG.data_dir); DATA_ROOT.mkdir(parents=True, exist_ok=True)\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nAMP_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16\n\nif torch.cuda.is_available():\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.backends.cudnn.allow_tf32 = True\n\nprint("device:", DEVICE)\nif torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0), "AMP:", AMP_DTYPE)\nprint("HAS_MAMBA:", HAS_MAMBA, "HAS_TRITON:", HAS_TRITON)\nif not HAS_MAMBA:\n    print("Mamba import error:", MAMBA_IMPORT_ERROR)\n    print(\'Install with: pip install "mamba-ssm[causal-conv1d]" --no-build-isolation\')\n\n# %%\n# ============================================================\n# 2. REPRODUCIBILITY / UTILS\n# ============================================================\n\ndef seed_all(seed: int):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\ndef n_params(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef init_embedding(emb: nn.Embedding):\n    nn.init.normal_(emb.weight, mean=0.0, std=0.02)\n    if emb.padding_idx is not None:\n        with torch.no_grad(): emb.weight[emb.padding_idx].zero_()\n\n\ndef cosine_lr(epoch: int, max_epochs: int, peak: float, minimum: float, warmup: int = 3):\n    if epoch <= warmup:\n        return peak * epoch / max(1, warmup)\n    p = (epoch - warmup) / max(1, max_epochs - warmup)\n    return minimum + (peak - minimum) * 0.5 * (1 + math.cos(math.pi * p))\n\n\ndef safe_name(s: str) -> str:\n    return s.lower().replace("+", "plus").replace("-", "_").replace(" ", "_")\n\n# %%\n# ============================================================\n# 3. DATA LOADING / PREPROCESSING\n# ============================================================\n\ndef download(url: str, path: Path):\n    if path.exists(): return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    print("Downloading", url, "->", path)\n    urllib.request.urlretrieve(url, path)\n\n\ndef _dedupe_consecutive(pairs: List[Tuple[int, int]]) -> Tuple[List[int], List[int]]:\n    items, times = [], []\n    last = None\n    for ts, item in sorted(pairs):\n        if item != last:\n            items.append(item); times.append(int(ts)); last = item\n    return items, times\n\n\ndef _finalize_events(raw_events: Dict[str, List[Tuple[int, str]]], min_user_len: int):\n    # Iterative user/item 5-core. Amazon inputs are already nominally 5-core,\n    # but doing it here standardizes all sources and protects after dedupe.\n    active = {u: list(v) for u, v in raw_events.items()}\n    changed = True\n    while changed:\n        changed = False\n        active = {u: ev for u, ev in active.items() if len(ev) >= min_user_len}\n        item_count = Counter(item for ev in active.values() for _, item in ev)\n        good = {i for i, c in item_count.items() if c >= min_user_len}\n        nxt = {}\n        for u, ev in active.items():\n            ev2 = [(t, i) for t, i in ev if i in good]\n            if len(ev2) >= min_user_len: nxt[u] = ev2\n        if len(nxt) != len(active) or any(len(nxt.get(u, [])) != len(ev) for u, ev in active.items()):\n            changed = True\n        active = nxt\n\n    all_items = sorted({i for ev in active.values() for _, i in ev})\n    item_map = {x: j + 1 for j, x in enumerate(all_items)}\n    seqs, tss = [], []\n    for u in sorted(active):\n        pairs = [(t, item_map[i]) for t, i in active[u]]\n        items, times = _dedupe_consecutive(pairs)\n        if len(items) >= min_user_len:\n            seqs.append(items); tss.append(times)\n    freq = Counter(i for s in seqs for i in s[:-2])\n    return {\n        "sequences": seqs,\n        "timestamps": tss,\n        "num_items": len(all_items),\n        "frequency": dict(freq),\n    }\n\n\ndef load_amazon(spec: DatasetSpec, cache: Path):\n    if cache.exists() and CFG.reuse:\n        return torch.load(cache, weights_only=False)\n    path = DATA_ROOT / spec.filename\n    download(spec.url, path)\n    events = defaultdict(list)\n    print("Parsing", spec.name)\n    with gzip.open(path, "rt", encoding="utf8") as f:\n        for line in f:\n            row = ast.literal_eval(line)\n            events[str(row["reviewerID"])].append((int(row["unixReviewTime"]), str(row["asin"])))\n    out = _finalize_events(events, CFG.min_user_len)\n    torch.save(out, cache)\n    return out\n\n\ndef _finalize_movielens_frame(df: pd.DataFrame, min_user_len: int):\n    """Memory-conscious iterative k-core + temporal sequence build for MovieLens."""\n    df = df[["user", "item", "timestamp"]].copy()\n    # Iterative 5-core. Use integer ids throughout to avoid millions of Python strings/tuples.\n    while True:\n        n0 = len(df)\n        uc = df["user"].value_counts(sort=False)\n        ic = df["item"].value_counts(sort=False)\n        good_u = uc.index[uc.values >= min_user_len]\n        good_i = ic.index[ic.values >= min_user_len]\n        df = df[df["user"].isin(good_u) & df["item"].isin(good_i)]\n        if len(df) == n0:\n            break\n    df = df.sort_values(["user", "timestamp"], kind="stable")\n    # Remove only consecutive repeated items, matching the Amazon path.\n    prev = df.groupby("user", sort=False)["item"].shift(1)\n    df = df[df["item"] != prev]\n    # One more user filter after dedupe.\n    uc = df["user"].value_counts(sort=False)\n    df = df[df["user"].isin(uc.index[uc.values >= min_user_len])]\n    # Dense remap 1..V after filtering.\n    item_codes, item_uniques = pd.factorize(df["item"], sort=True)\n    df = df.assign(item_new=(item_codes.astype(np.int64) + 1))\n    seqs, tss = [], []\n    for _, g in df.groupby("user", sort=True):\n        if len(g) >= min_user_len:\n            seqs.append(g["item_new"].astype(np.int64).tolist())\n            tss.append(g["timestamp"].astype(np.int64).tolist())\n    freq = Counter(i for seq in seqs for i in seq[:-2])\n    return {"sequences": seqs, "timestamps": tss, "num_items": int(len(item_uniques)), "frequency": dict(freq)}\n\n\ndef load_ml1m(spec: DatasetSpec, cache: Path):\n    if cache.exists() and CFG.reuse:\n        return torch.load(cache, weights_only=False)\n    zpath = DATA_ROOT / spec.filename\n    download(spec.url, zpath)\n    with zipfile.ZipFile(zpath) as z:\n        # pandas\' Python engine is required for the multi-character :: separator.\n        df = pd.read_csv(\n            z.open(spec.archive_member), sep="::", engine="python", header=None,\n            names=["user", "item", "rating", "timestamp"],\n            usecols=[0, 1, 3], dtype={"user":"int32", "item":"int32", "timestamp":"int64"},\n        )\n    out = _finalize_movielens_frame(df, CFG.min_user_len)\n    del df; gc.collect()\n    torch.save(out, cache)\n    return out\n\n\ndef load_ml20m(spec: DatasetSpec, cache: Path):\n    if cache.exists() and CFG.reuse:\n        return torch.load(cache, weights_only=False)\n    zpath = DATA_ROOT / spec.filename\n    download(spec.url, zpath)\n    with zipfile.ZipFile(zpath) as z:\n        df = pd.read_csv(\n            z.open(spec.archive_member), usecols=["userId", "movieId", "timestamp"],\n            dtype={"userId":"int32", "movieId":"int32", "timestamp":"int64"},\n        ).rename(columns={"userId":"user", "movieId":"item"})\n    out = _finalize_movielens_frame(df, CFG.min_user_len)\n    del df; gc.collect()\n    torch.save(out, cache)\n    return out\n\n\ndef load_dataset(name: str):\n    spec = DATASETS[name]\n    cache = DATA_ROOT / f"{name}_core{CFG.min_user_len}_cache.pt"\n    if spec.kind == "amazon": out = load_amazon(spec, cache)\n    elif spec.kind == "ml1m": out = load_ml1m(spec, cache)\n    elif spec.kind == "ml20m": out = load_ml20m(spec, cache)\n    else: raise ValueError(spec.kind)\n    seqs = out["sequences"]\n    stats = {\n        "dataset": name, "users": len(seqs), "items": out["num_items"],\n        "interactions": int(sum(map(len, seqs))),\n        "avg_len": float(np.mean(list(map(len, seqs)))),\n        "median_len": float(np.median(list(map(len, seqs)))),\n        "p95_len": float(np.percentile(list(map(len, seqs)), 95)),\n        "max_len": int(max(map(len, seqs))),\n    }\n    print(stats)\n    return out, stats\n\n\ndef split_data(data):\n    seqs, tss = data["sequences"], data["timestamps"]\n    train = [s[:-2] for s in seqs]\n    val_prefix = [s[:-2] for s in seqs]\n    val_target = [s[-2] for s in seqs]\n    test_prefix = [s[:-1] for s in seqs]\n    test_target = [s[-1] for s in seqs]\n    train_ts = [t[:-2] for t in tss]\n    return train, train_ts, val_prefix, val_target, test_prefix, test_target\n\n# %%\n# ============================================================\n# 4. RANDOM-WINDOW TRAIN DATA\n# ============================================================\n\nclass WindowDataset(Dataset):\n    def __init__(self, train_seqs: List[List[int]], max_len: int, seed: int):\n        self.seqs = [s for s in train_seqs if len(s) >= 2]\n        self.max_len = max_len\n        self.seed = seed\n        self.epoch = 0\n    def set_epoch(self, epoch: int): self.epoch = epoch\n    def __len__(self): return len(self.seqs)\n    def __getitem__(self, idx):\n        s = self.seqs[idx]\n        # One deterministic random window per user per epoch. Over epochs this\n        # covers long user histories instead of always keeping only the tail.\n        rng = random.Random(self.seed * 1_000_003 + self.epoch * 97_409 + idx)\n        if len(s) <= self.max_len + 1:\n            end = len(s)\n        else:\n            end = rng.randint(2, len(s))\n        start = max(0, end - (self.max_len + 1))\n        seg = s[start:end]\n        if len(seg) < 2:\n            seg = s[:2]\n        return torch.tensor(seg, dtype=torch.long)\n\n\ndef collate_windows(batch):\n    L = max(x.numel() for x in batch)\n    out = torch.zeros(len(batch), L, dtype=torch.long)\n    lengths = torch.empty(len(batch), dtype=torch.long)\n    for r, x in enumerate(batch):\n        out[r, :x.numel()] = x\n        lengths[r] = x.numel()\n    return out, lengths\n\n\ndef make_train_loader(ds: WindowDataset, epoch: int, batch_size: int):\n    ds.set_epoch(epoch)\n    g = torch.Generator(); g.manual_seed(ds.seed + epoch)\n    return DataLoader(ds, batch_size=batch_size, shuffle=True, generator=g,\n                      collate_fn=collate_windows, num_workers=CFG.num_workers,\n                      pin_memory=torch.cuda.is_available())\n\n# %%\n# ============================================================\n# 5. COMMON LOSSES / BASE MODEL API\n# ============================================================\n\nclass ARRecommender(nn.Module):\n    model_name = "AR"\n    def __init__(self, n_items: int, max_len: int, d_model: int):\n        super().__init__(); self.n_items = n_items; self.max_len = max_len; self.d_model = d_model\n    @property\n    def item_weight(self): raise NotImplementedError\n    def encode(self, seq: torch.Tensor) -> torch.Tensor: raise NotImplementedError\n    def score_hidden(self, h: torch.Tensor) -> torch.Tensor:\n        z = h @ self.item_weight[:self.n_items + 1].T\n        z[..., 0] = -1e9\n        return z\n    def last_hidden(self, seq: torch.Tensor, lengths: torch.Tensor):\n        H = self.encode(seq)\n        rows = torch.arange(seq.size(0), device=seq.device)\n        return H[rows, lengths - 1]\n    def full_scores(self, seq: torch.Tensor, lengths: torch.Tensor):\n        return self.score_hidden(self.last_hidden(seq, lengths))\n\n\ndef autoregressive_inputs(tokens: torch.Tensor, lengths: torch.Tensor):\n    # tokens contains >=2 real items per row and right padding.\n    x = tokens[:, :-1]\n    y = tokens[:, 1:]\n    return x, y\n\n\ndef sampled_softmax_from_hidden(\n    hidden, target, item_weight, n_items, n_negs=256,\n    normalize=False, temperature=1.0, eps=1e-6,\n):\n    valid = target != 0\n    h = hidden[valid]\n    y = target[valid]\n    if h.numel() == 0:\n        return hidden.sum() * 0\n    neg = torch.randint(1, n_items + 1, (n_negs,), device=h.device)\n    if normalize:\n        h = F.normalize(h.float(), dim=-1, eps=eps)\n        pos_w = F.normalize(item_weight[y].float(), dim=-1, eps=eps)\n        neg_w = F.normalize(item_weight[neg].float(), dim=-1, eps=eps)\n        pos = (h * pos_w).sum(-1, keepdim=True)\n        neg_logits = h @ neg_w.T\n    else:\n        pos = (h * item_weight[y]).sum(-1, keepdim=True)\n        neg_logits = h @ item_weight[neg].T\n    pos = pos / temperature\n    neg_logits = neg_logits / temperature\n    # Avoid treating a sampled copy of the positive as a negative.\n    neg_logits = neg_logits.masked_fill(neg[None, :] == y[:, None], -1e9)\n    logits = torch.cat([pos, neg_logits], dim=1)\n    labels = torch.zeros(h.size(0), dtype=torch.long, device=h.device)\n    return F.cross_entropy(logits, labels)\n\n\ndef ar_training_loss(model: ARRecommender, tokens, lengths, loss_mode="full", n_negs=256):\n    x, y = autoregressive_inputs(tokens, lengths)\n    H = model.encode(x)\n    valid = y != 0\n    if loss_mode == "hstu_ss":\n        return sampled_softmax_from_hidden(\n            H, y, model.item_weight, model.n_items, n_negs,\n            normalize=True,\n            temperature=CFG.hstu_temperature,\n            eps=CFG.hstu_l2_eps,\n        )\n    if loss_mode == "ss":\n        return sampled_softmax_from_hidden(H, y, model.item_weight, model.n_items, n_negs)\n    return F.cross_entropy(model.score_hidden(H[valid]), y[valid])\n\n# %%\n# ============================================================\n# 6. SASREC + eSASRec\n# ============================================================\n\nclass FFN(nn.Module):\n    def __init__(self, d, inner, dropout, swiglu=False):\n        super().__init__(); self.swiglu = swiglu\n        if swiglu:\n            self.up = nn.Linear(d, inner, bias=False); self.gate = nn.Linear(d, inner, bias=False)\n            self.down = nn.Linear(inner, d, bias=False)\n        else:\n            self.net = nn.Sequential(nn.Linear(d, inner), nn.GELU(), nn.Dropout(dropout), nn.Linear(inner, d))\n        self.drop = nn.Dropout(dropout)\n    def forward(self, x):\n        if self.swiglu: return self.drop(self.down(F.silu(self.up(x)) * self.gate(x)))\n        return self.net(x)\n\nclass SASBlock(nn.Module):\n    def __init__(self, d, heads, inner, dropout, ligr=False):\n        super().__init__(); self.ligr = ligr\n        self.n1 = nn.LayerNorm(d); self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)\n        self.n2 = nn.LayerNorm(d); self.ffn = FFN(d, inner, dropout, swiglu=ligr)\n        self.d1 = nn.Dropout(dropout); self.d2 = nn.Dropout(dropout)\n        if ligr:\n            self.g1 = nn.Linear(d, d, bias=False); self.g2 = nn.Linear(d, d, bias=False)\n    def forward(self, x, padding):\n        L = x.size(1)\n        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), diagonal=1)\n        z = self.n1(x)\n        a, _ = self.attn(z, z, z, attn_mask=causal, key_padding_mask=padding, need_weights=False)\n        a = self.d1(a)\n        x = x + (a * torch.sigmoid(self.g1(x)) if self.ligr else a)\n        f = self.d2(self.ffn(self.n2(x)))\n        return x + (f * torch.sigmoid(self.g2(x)) if self.ligr else f)\n\nclass SASRecModel(ARRecommender):\n    def __init__(self, n_items, max_len, d=64, layers=2, heads=2, inner=256, dropout=.2, ligr=False):\n        super().__init__(n_items, max_len, d)\n        self.item = nn.Embedding(n_items + 2, d, padding_idx=0); init_embedding(self.item)\n        self.pos = nn.Embedding(max_len, d); init_embedding(self.pos)\n        self.drop = nn.Dropout(dropout)\n        self.blocks = nn.ModuleList([SASBlock(d, heads, inner, dropout, ligr=ligr) for _ in range(layers)])\n        self.norm = nn.LayerNorm(d)\n        for m in self.modules():\n            if isinstance(m, nn.Linear):\n                nn.init.xavier_uniform_(m.weight)\n                if m.bias is not None: nn.init.zeros_(m.bias)\n    @property\n    def item_weight(self): return self.item.weight\n    def encode(self, seq):\n        B, L = seq.shape\n        p = torch.arange(L, device=seq.device)[None]\n        x = self.drop(self.item(seq) + self.pos(p))  # clean path: no sqrt(d)\n        padding = seq == 0\n        for b in self.blocks: x = b(x, padding)\n        return self.norm(x)\n\n# %%\n# ============================================================\n# 7. GRU4REC\n# ============================================================\n\nclass GRU4RecModel(ARRecommender):\n    def __init__(self, n_items, max_len, d=64, layers=2, dropout=.2):\n        super().__init__(n_items, max_len, d)\n        self.item = nn.Embedding(n_items + 2, d, padding_idx=0); init_embedding(self.item)\n        self.drop = nn.Dropout(dropout)\n        self.gru = nn.GRU(d, d, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0)\n        self.norm = nn.LayerNorm(d)\n    @property\n    def item_weight(self): return self.item.weight\n    def encode(self, seq):\n        x = self.drop(self.item(seq))\n        h, _ = self.gru(x)\n        return self.norm(h)\n\n# %%\n# ============================================================\n# 8. HSTU-CORE\n# ============================================================\n\nclass HSTUBlock(nn.Module):\n    """ID-only HSTU block following Meta\'s public SequentialTransductionUnit math.\n\n    Official core:\n      norm_x = LN(x)\n      [U,V,Q,K] = SiLU(norm_x @ W_uvqk)\n      A = SiLU(Q K^T + relative_bias) / n\n      y = W_o(dropout(U * LN(A V))) + x\n\n    We retain relative *position* bias but omit timestamp bias so every model in\n    the controlled track remains item-ID-only.\n    """\n    def __init__(self, d, heads, dqk, dv, dropout, max_len, eps=1e-6):\n        super().__init__()\n        self.d=d; self.h=heads; self.dqk=dqk; self.dv=dv\n        self.max_len=max_len; self.eps=eps\n        out_dim = 2 * heads * dv + 2 * heads * dqk\n        self.uvqk = nn.Parameter(torch.empty(d, out_dim).normal_(mean=0.0, std=0.02))\n        self.o = nn.Linear(heads * dv, d)\n        nn.init.xavier_uniform_(self.o.weight)\n        if self.o.bias is not None:\n            nn.init.zeros_(self.o.bias)\n        self.drop = nn.Dropout(dropout)\n\n        # Official RelativePositionalBias is shared across heads.\n        self.rel_w = nn.Parameter(torch.empty(2 * max_len - 1).normal_(mean=0.0, std=0.02))\n        pos=torch.arange(max_len)\n        rel=(pos[:,None]-pos[None,:]).clamp(-(max_len-1),max_len-1)+(max_len-1)\n        self.register_buffer("rel_index", rel, persistent=False)\n        causal=torch.triu(torch.ones(max_len,max_len,dtype=torch.bool),diagonal=1)\n        self.register_buffer("causal", causal, persistent=False)\n\n    def _ln(self,x,dim):\n        return F.layer_norm(x, normalized_shape=[dim], eps=self.eps)\n\n    def forward(self, x, padding):\n        B,L,D=x.shape\n        xn=self._ln(x,D)\n        proj=F.silu(xn @ self.uvqk)\n\n        uv = self.h * self.dv\n        qk = self.h * self.dqk\n        u,v,q,k = torch.split(proj,[uv,uv,qk,qk],dim=-1)\n\n        u=u.view(B,L,self.h,self.dv)\n        v=v.view(B,L,self.h,self.dv)\n        q=q.view(B,L,self.h,self.dqk)\n        k=k.view(B,L,self.h,self.dqk)\n\n        # IMPORTANT: HSTU does NOT use Transformer 1/sqrt(d) scaling here.\n        attn=torch.einsum("bnhd,bmhd->bhnm",q,k)\n        rb=self.rel_w[self.rel_index[:L,:L]][None,None,:,:]\n        attn=F.silu(attn + rb)\n\n        valid=(~self.causal[:L,:L])[None,None,:,:]\n        valid=valid & (~padding[:,None,None,:])\n        attn=attn.masked_fill(~valid,0.0)\n\n        # Meta implementation divides by attention tensor length n.\n        # Because this harness dynamically pads only to the batch max, n == L.\n        attn=attn / max(1,L)\n\n        a=torch.einsum("bhnm,bmhd->bnhd",attn,v).reshape(B,L,self.h*self.dv)\n        a=self._ln(a,self.h*self.dv)\n        o_in=(u.reshape(B,L,self.h*self.dv) * a)\n        y=self.o(self.drop(o_in))\n        out=x+y\n        return out.masked_fill(padding[:,:,None],0.0)\n\n\nclass HSTUModel(ARRecommender):\n    def __init__(\n        self,n_items,max_len,d=64,layers=4,heads=4,dqk=16,dv=16,\n        dropout=.5,l2_eps=1e-6,\n    ):\n        super().__init__(n_items,max_len,d)\n        self.item=nn.Embedding(n_items+2,d,padding_idx=0)\n        init_embedding(self.item)\n        self.pos=nn.Embedding(max_len,d)\n        init_embedding(self.pos)\n        self.drop=nn.Dropout(dropout)\n        self.blocks=nn.ModuleList([\n            HSTUBlock(d,heads,dqk,dv,dropout,max_len,l2_eps)\n            for _ in range(layers)\n        ])\n        self.l2_eps=l2_eps\n\n    @property\n    def item_weight(self):\n        return self.item.weight\n\n    def encode(self,seq):\n        B,L=seq.shape\n        p=torch.arange(L,device=seq.device)[None]\n        x=self.drop(self.item(seq)+self.pos(p))\n        pad=seq==0\n        for b in self.blocks:\n            x=b(x,pad)\n        # Meta public configs use L2NormEmbeddingPostprocessor.\n        return F.normalize(x.float(),dim=-1,eps=self.l2_eps).to(x.dtype)\n\n    def score_hidden(self,h):\n        q=F.normalize(h.float(),dim=-1,eps=self.l2_eps)\n        w=F.normalize(self.item.weight[:self.n_items+1].float(),dim=-1,eps=self.l2_eps)\n        z=q@w.T\n        z[...,0]=-1e9\n        return z\n\n\ndef hstu_architecture(dataset_name, large=False):\n    """Closest public Meta recipe by dataset family.\n\n    Beauty/Video/Sports/Toys use the public Amazon-Books L=50 HSTU shape\n    because Meta does not publish a separate Beauty HSTU config.\n    """\n    if dataset_name == "ml1m":\n        return dict(\n            d=50,\n            layers=8 if large else 2,\n            heads=2 if large else 1,\n            dqk=25 if large else 50,\n            dv=25 if large else 50,\n            dropout=.2,\n        )\n    if dataset_name == "ml20m":\n        return dict(\n            d=256,\n            layers=16 if large else 4,\n            heads=8 if large else 4,\n            dqk=32 if large else 64,\n            dv=32 if large else 64,\n            dropout=.2,\n        )\n    # Public Amazon Books L=50 recipe, used as our Amazon-50 HSTU recipe.\n    return dict(\n        d=64,\n        layers=16 if large else 4,\n        heads=8 if large else 4,\n        dqk=8 if large else 16,\n        dv=8 if large else 16,\n        dropout=.5,\n    )\n\n# %%\n# ============================================================\n# 9. FMLP-REC\n# ============================================================\n\nclass FMLPFilterBlock(nn.Module):\n    """FMLP-Rec filter block, following the public implementation:\n    FFT along sequence -> learned complex filter -> residual LayerNorm ->\n    GELU FFN -> residual LayerNorm.\n    """\n    def __init__(self, max_len, d, inner, dropout):\n        super().__init__()\n        self.real = nn.Parameter(torch.randn(1, max_len//2 + 1, d) * 0.02)\n        self.imag = nn.Parameter(torch.randn(1, max_len//2 + 1, d) * 0.02)\n        self.filter_drop = nn.Dropout(dropout)\n        self.filter_ln = nn.LayerNorm(d, eps=1e-12)\n        self.ff1 = nn.Linear(d, inner)\n        self.ff2 = nn.Linear(inner, d)\n        self.ff_drop = nn.Dropout(dropout)\n        self.ff_ln = nn.LayerNorm(d, eps=1e-12)\n\n    def forward(self, x, pad):\n        B,L,D = x.shape\n        xf = torch.fft.rfft(x.float(), dim=1, norm="ortho")\n        w = torch.complex(self.real[:, :xf.size(1)], self.imag[:, :xf.size(1)])\n        y = torch.fft.irfft(xf * w, n=L, dim=1, norm="ortho").to(x.dtype)\n        h = self.filter_ln(x + self.filter_drop(y))\n        f = self.ff2(self.ff_drop(F.gelu(self.ff1(h))))\n        h = self.ff_ln(h + self.ff_drop(f))\n        return h.masked_fill(pad[:,:,None], 0.0)\n\nclass FMLPRecModel(ARRecommender):\n    def __init__(self, n_items, max_len, d=64, layers=2, inner=256, dropout=.2):\n        super().__init__(n_items, max_len, d)\n        self.item = nn.Embedding(n_items + 2, d, padding_idx=0); init_embedding(self.item)\n        self.pos = nn.Embedding(max_len, d); init_embedding(self.pos)\n        self.in_ln = nn.LayerNorm(d, eps=1e-12)\n        self.drop = nn.Dropout(dropout)\n        self.blocks = nn.ModuleList([FMLPFilterBlock(max_len, d, inner, dropout) for _ in range(layers)])\n    @property\n    def item_weight(self): return self.item.weight\n    def encode(self, seq):\n        B,L = seq.shape; p = torch.arange(L, device=seq.device)[None]\n        x = self.drop(self.in_ln(self.item(seq) + self.pos(p))); pad = seq == 0\n        for b in self.blocks: x = b(x, pad)\n        return x\n\n# %%\n# ============================================================\n# 10. MAMBA4REC\n# ============================================================\n\nclass MambaRecLayer(nn.Module):\n    def __init__(self, d, d_state, d_conv, expand, dropout, n_layers):\n        super().__init__()\n        if not HAS_MAMBA:\n            raise RuntimeError(\'mamba_ssm is not installed. Run: pip install "mamba-ssm[causal-conv1d]" --no-build-isolation\')\n        self.n_layers = n_layers\n        self.mamba = Mamba(d_model=d, d_state=d_state, d_conv=d_conv, expand=expand)\n        self.drop = nn.Dropout(dropout); self.norm = nn.LayerNorm(d, eps=1e-12)\n        self.ff1 = nn.Linear(d, 4*d); self.ff2 = nn.Linear(4*d, d)\n        self.ffdrop = nn.Dropout(dropout); self.ffnorm = nn.LayerNorm(d, eps=1e-12)\n    def forward(self, x):\n        h = self.mamba(x)\n        h = self.norm(self.drop(h) + x) if self.n_layers > 1 else self.norm(self.drop(h))\n        f = self.ff2(self.ffdrop(F.gelu(self.ff1(h)))); f = self.ffdrop(f)\n        return self.ffnorm(f + h)\n\nclass Mamba4RecModel(ARRecommender):\n    def __init__(self, n_items, max_len, d=64, layers=2, dropout=.2, d_state=32, d_conv=4, expand=2):\n        super().__init__(n_items, max_len, d)\n        self.item = nn.Embedding(n_items + 2, d, padding_idx=0); init_embedding(self.item)\n        self.in_norm = nn.LayerNorm(d, eps=1e-12); self.drop = nn.Dropout(dropout)\n        self.blocks = nn.ModuleList([MambaRecLayer(d,d_state,d_conv,expand,dropout,layers) for _ in range(layers)])\n        self.apply(self._init)\n        with torch.no_grad(): self.item.weight[0].zero_()\n    def _init(self, m):\n        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0, 0.02)\n        if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)\n        if isinstance(m, nn.LayerNorm): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)\n    @property\n    def item_weight(self): return self.item.weight\n    def encode(self, seq):\n        x = self.in_norm(self.drop(self.item(seq)))\n        for b in self.blocks: x = b(x)\n        return x.masked_fill((seq==0)[:,:,None], 0.0)\n\n# %%\n# ============================================================\n# 11. BERT4REC\n# ============================================================\n\nclass BERT4RecModel(nn.Module):\n    def __init__(self, n_items, max_len, d=64, layers=2, heads=2, inner=256, dropout=.2):\n        super().__init__(); self.n_items=n_items; self.max_len=max_len; self.d_model=d\n        self.mask_id = n_items + 1\n        self.item = nn.Embedding(n_items + 2, d, padding_idx=0); init_embedding(self.item)\n        self.pos = nn.Embedding(max_len, d); init_embedding(self.pos)\n        layer = nn.TransformerEncoderLayer(d, heads, inner, dropout, activation="gelu", batch_first=True, norm_first=True)\n        self.enc = nn.TransformerEncoder(layer, layers); self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)\n    @property\n    def item_weight(self): return self.item.weight\n    def encode(self, seq):\n        B,L=seq.shape; p=torch.arange(L,device=seq.device)[None]\n        x=self.drop(self.item(seq)+self.pos(p)); x=self.enc(x, src_key_padding_mask=(seq==0)); return self.norm(x)\n    def masked_loss(self, tokens, lengths, mask_prob=.2, loss_mode="full", n_negs=256):\n        # Use at most max_len real tokens; random Cloze masks. Ensure >=1 mask/user.\n        x=tokens[:,-self.max_len:].clone(); real=x!=0\n        rand=torch.rand_like(x.float()); mask=(rand<mask_prob)&real\n        for r in range(x.size(0)):\n            idx=real[r].nonzero(as_tuple=False).flatten()\n            if idx.numel() and not mask[r].any(): mask[r, idx[torch.randint(idx.numel(),(1,),device=x.device)]]=True\n        target=x.clone(); inp=x.clone();\n        # Standard BERT-style 80/10/10 corruption on selected positions.\n        u=torch.rand_like(inp.float())\n        inp[mask & (u<0.8)] = self.mask_id\n        random_mask = mask & (u>=0.8) & (u<0.9)\n        inp[random_mask] = torch.randint(1,self.n_items+1,(int(random_mask.sum()),),device=x.device)\n        H=self.encode(inp); h=H[mask]; y=target[mask]\n        if loss_mode=="ss":\n            return sampled_softmax_from_hidden(H, target.masked_fill(~mask,0), self.item_weight, self.n_items, n_negs)\n        logits=h @ self.item.weight[:self.n_items+1].T; logits[:,0]=-1e9\n        return F.cross_entropy(logits,y)\n    def full_scores(self, seq, lengths):\n        B=seq.size(0)\n        out=[]\n        for r in range(B):\n            n=int(lengths[r].item()); hist=seq[r,:n]\n            hist=hist[-(self.max_len-1):]\n            z=torch.zeros(self.max_len,dtype=torch.long,device=seq.device)\n            z[:hist.numel()]=hist; z[hist.numel()]=self.mask_id\n            out.append(z)\n        z=torch.stack(out); H=self.encode(z)\n        idx=torch.tensor([min(int(l.item()),self.max_len-1) for l in lengths],device=seq.device)\n        rows=torch.arange(B,device=seq.device); h=H[rows,idx]\n        scores=h @ self.item.weight[:self.n_items+1].T; scores[:,0]=-1e9\n        return scores\n\n# %%\n# ============================================================\n# 12. PERSISTENT SPARSE WALKER\n# ============================================================\n\nclass ConceptSpace(nn.Module):\n    def __init__(self, d, side, h):\n        super().__init__(); self.side=side; self.h=h\n        self.left_router=nn.Parameter(torch.randn(side,h)/math.sqrt(h)); self.right_router=nn.Parameter(torch.randn(side,h)/math.sqrt(h))\n        self.left_key=nn.Parameter(torch.randn(side,h)/math.sqrt(h)); self.right_key=nn.Parameter(torch.randn(side,h)/math.sqrt(h))\n        self.left_value=nn.Embedding(side,d); self.right_value=nn.Embedding(side,d); init_embedding(self.left_value); init_embedding(self.right_value)\n        self.value_proj=nn.Linear(2*d,d)\n    def split(self, ids): return ids//self.side, ids%self.side\n    def value(self, ids):\n        l,r=self.split(ids); return self.value_proj(torch.cat([self.left_value(l),self.right_value(r)],-1))\n    def key(self, ids):\n        l,r=self.split(ids); return F.normalize(self.left_key[l]+self.right_key[r],dim=-1)\n\nclass Router(nn.Module):\n    def __init__(self,d,h,top_side,side):\n        super().__init__(); self.top_side=top_side; self.side=side\n        self.left_q=nn.Linear(d,h,bias=False); self.right_q=nn.Linear(d,h,bias=False); self.scale=nn.Parameter(torch.tensor(math.log(10.0)))\n    def forward(self, hidden, space):\n        ql=F.normalize(self.left_q(hidden),dim=-1); qr=F.normalize(self.right_q(hidden),dim=-1)\n        kl=F.normalize(space.left_router,dim=-1); kr=F.normalize(space.right_router,dim=-1); sc=torch.exp(self.scale)\n        lv,li=(ql@kl.T*sc).topk(self.top_side,-1); rv,ri=(qr@kr.T*sc).topk(self.top_side,-1)\n        ids=(li.unsqueeze(-1)*self.side+ri.unsqueeze(-2)).reshape(hidden.size(0),-1)\n        logits=(lv.unsqueeze(-1)+rv.unsqueeze(-2)).reshape(hidden.size(0),-1)\n        return ids,F.softmax(logits,-1)\n\nclass CompactGraph(nn.Module):\n    def __init__(self,d,h,n_concepts,degree,active):\n        super().__init__(); self.n_concepts=n_concepts; self.degree=degree; self.active=active\n        self.edge_logits=nn.Embedding(n_concepts,degree); nn.init.normal_(self.edge_logits.weight,std=.02)\n        dest=torch.randint(0,n_concepts,(n_concepts,degree),dtype=torch.int32); dest[:,0]=torch.arange(n_concepts,dtype=torch.int32)\n        self.register_buffer("destination",dest); self.register_buffer("touched",torch.zeros(n_concepts,dtype=torch.bool),persistent=False)\n        self.context_q=nn.Linear(d,h,bias=False); self.scale=nn.Parameter(torch.tensor(math.log(3.0)))\n    def topk(self, ids, mass):\n        v,slot=mass.topk(self.active,-1); ids=ids.gather(-1,slot); v=v/(v.sum(-1,keepdim=True)+1e-8); return ids,v\n    def forward(self,ids,mass,context,space):\n        if self.training: self.touched[ids.detach().reshape(-1)]=True\n        dest=self.destination[ids].long(); static=self.edge_logits(ids); q=F.normalize(self.context_q(context),dim=-1); key=space.key(dest)\n        score=static+torch.exp(self.scale)*(key*q[:,None,None,:]).sum(-1); prob=F.softmax(score,-1)\n        B=ids.size(0); return self.topk(dest.reshape(B,-1),(mass.unsqueeze(-1)*prob).reshape(B,-1))\n    @torch.no_grad()\n    def pursue(self,opt,refresh):\n        rows=self.touched.nonzero(as_tuple=False).squeeze(-1)\n        if rows.numel()==0:return 0\n        w=self.edge_logits.weight[rows]; repl=w.abs().argsort(-1)[:,:refresh]; rr=rows[:,None].expand_as(repl)\n        self.destination[rr,repl]=torch.randint(0,self.n_concepts,repl.shape,device=self.destination.device,dtype=torch.int32)\n        self.edge_logits.weight[rr,repl]=0\n        st=opt.state.get(self.edge_logits.weight,{})\n        for k in ("exp_avg","exp_avg_sq"):\n            if st.get(k) is not None: st[k][rr,repl]=0\n        n=int(rows.numel()); self.touched.zero_(); return n\n\nclass SparseWalkerModel(ARRecommender):\n    def __init__(self,n_items,max_len,d=64,layers=2,side=256,h=16,active=8,top_side=2,degree=4,fresh_weight=.25):\n        super().__init__(n_items,max_len,d); self.layers_n=layers; self.side=side; self.h=h; self.active=active; self.degree=degree; self.fresh_weight=fresh_weight\n        self.n_concepts=side*side; self.fresh_concepts=top_side*top_side\n        self.item=nn.Embedding(n_items+1,d,padding_idx=0); init_embedding(self.item)\n        self.space=ConceptSpace(d,side,h); self.router=Router(d,h,top_side,side); self.graph=CompactGraph(d,h,self.n_concepts,degree,active)\n        self.message_proj=nn.Linear(d,d,bias=False); self.norm=nn.LayerNorm(d)\n    @property\n    def item_weight(self):return self.item.weight\n    def _top(self,ids,mass):\n        v,s=mass.topk(self.active,-1); ids=ids.gather(-1,s); v=v/(v.sum(-1,keepdim=True)+1e-8); return ids,v\n    def _merge(self,oi,om,fi,fm):\n        return self._top(torch.cat([oi,fi],-1),torch.cat([(1-self.fresh_weight)*om,self.fresh_weight*fm],-1))\n    def encode_with_states(self,seq):\n        B,L=seq.shape; valid=seq!=0; item_state=self.item(seq)*math.sqrt(self.d_model)\n        fi,fm=self.router(item_state.reshape(B*L,self.d_model),self.space); fi=fi.view(B,L,-1); fm=fm.view(B,L,-1)\n        ids=torch.zeros(B,self.active,dtype=torch.long,device=seq.device); mass=torch.zeros(B,self.active,dtype=item_state.dtype,device=seq.device)\n        outs=[]; ids_hist=[]; mass_hist=[]\n        for t in range(L):\n            act=valid[:,t]; af=act.to(item_state.dtype)[:,None]; xids=ids; xmass=mass*af; fids=fi[:,t]; fmass=fm[:,t]*af\n            for _ in range(self.layers_n):\n                xids,xmass=self._merge(xids,xmass,fids,fmass); xids,xmass=self.graph(xids,xmass,item_state[:,t],self.space)\n            ids=torch.where(act[:,None],xids,ids); mass=torch.where(act[:,None],xmass,mass)\n            msg=(self.space.value(ids)*mass[:,:,None]).sum(1); h=self.norm(item_state[:,t]+self.message_proj(msg))*af\n            outs.append(h); ids_hist.append(ids); mass_hist.append(mass)\n        return torch.stack(outs,1),torch.stack(ids_hist,1),torch.stack(mass_hist,1)\n    def encode(self,seq): return self.encode_with_states(seq)[0]\n\n\nclass SparseWalkerE2EModel(SparseWalkerModel):\n    """Persistent Sparse Walker with supervised sparse concept->item terminal memory.\n\n    Continuous parameters are trained from a loss over the currently reachable\n    terminal candidates plus the positive label. The discrete terminal topology\n    is refreshed at epoch boundaries from true next-item evidence weighted by\n    active concept mass ("terminal pursuit"). No dense teacher is used.\n    """\n    def __init__(self,n_items,max_len,d=64,layers=2,side=256,h=16,active=8,\n                 top_side=2,degree=4,fresh_weight=.25,terminal_degree=128):\n        super().__init__(n_items,max_len,d,layers,side,h,active,top_side,degree,fresh_weight)\n        self.terminal_degree=int(terminal_degree)\n        init=torch.randint(1,n_items+1,(self.n_concepts,self.terminal_degree),dtype=torch.int32)\n        self.register_buffer("terminal_items",init,persistent=True)\n        self._evidence_pairs=[]\n        self._evidence_weights=[]\n\n    @torch.no_grad()\n    def initialize_terminal(self,popular_items):\n        pop=torch.as_tensor(popular_items,dtype=torch.long,device=self.terminal_items.device)\n        pop=pop[(pop>0)&(pop<=self.n_items)]\n        if pop.numel()==0:\n            pop=torch.arange(1,min(self.n_items,self.terminal_degree)+1,device=self.terminal_items.device)\n        if pop.numel()<self.terminal_degree:\n            pop=pop.repeat(math.ceil(self.terminal_degree/max(1,pop.numel())))\n        row=pop[:self.terminal_degree].to(torch.int32)\n        self.terminal_items.copy_(row[None,:].expand(self.n_concepts,-1))\n\n    def begin_terminal_epoch(self):\n        self._evidence_pairs=[]\n        self._evidence_weights=[]\n\n    @torch.no_grad()\n    def _record_terminal_evidence(self,ids,mass,target,max_rows):\n        n=target.numel()\n        if n==0: return\n        if n>max_rows:\n            pick=torch.randperm(n,device=target.device)[:max_rows]\n            ids=ids[pick]; mass=mass[pick]; target=target[pick]\n        pair=ids.long()*(self.n_items+1)+target[:,None].long()\n        self._evidence_pairs.append(pair.reshape(-1).cpu())\n        self._evidence_weights.append(mass.float().reshape(-1).cpu())\n\n    def sparse_training_loss(self,tokens,evidence_per_batch=2048,chunk=256):\n        x,y=autoregressive_inputs(tokens,None)\n        H,I,M=self.encode_with_states(x)\n        valid=y!=0\n        h=H[valid]; ids=I[valid]; mass=M[valid]; tgt=y[valid]\n        if h.numel()==0:\n            return H.sum()*0\n        self._record_terminal_evidence(ids.detach(),mass.detach(),tgt.detach(),evidence_per_batch)\n\n        total=h.new_zeros((),dtype=torch.float32)\n        count=0\n        item_w=self.item.weight\n        for st in range(0,h.size(0),chunk):\n            en=min(h.size(0),st+chunk)\n            hh=h[st:en]; ii=ids[st:en]; yy=tgt[st:en]\n            cand=self.terminal_items[ii].long().reshape(en-st,-1)\n            cs=(item_w[cand]*hh[:,None,:]).sum(-1).float()\n\n            # Dense-shaped accumulator, but only reachable items receive finite scores.\n            # This preserves exact duplicate aggregation while the actual dot products\n            # are computed only for sparse terminal candidates.\n            table=torch.full((en-st,self.n_items+1),-1e9,device=hh.device,dtype=torch.float32)\n            table=table.scatter_reduce(1,cand,cs,reduce="amax",include_self=True)\n\n            # Positive injection is a training-only device when the target is not\n            # already reachable. Inference never receives this injection.\n            pos=(hh*item_w[yy]).sum(-1).float()\n            table=table.scatter_reduce(1,yy[:,None],pos[:,None],reduce="amax",include_self=True)\n\n            total=total+F.cross_entropy(table[:,1:],yy-1,reduction="sum")\n            count+=en-st\n        return total/max(1,count)\n\n    @torch.no_grad()\n    def refresh_terminal(self):\n        if not self._evidence_pairs:\n            return 0\n        pairs=torch.cat(self._evidence_pairs)\n        weights=torch.cat(self._evidence_weights).float()\n\n        order=torch.argsort(pairs)\n        pairs=pairs[order]; weights=weights[order]\n        up,inv=torch.unique_consecutive(pairs,return_inverse=True)\n        agg=torch.zeros(up.numel(),dtype=torch.float32)\n        agg.scatter_add_(0,inv,weights)\n\n        concept=up//(self.n_items+1)\n        item=up%(self.n_items+1)\n\n        # Descending evidence, then stable grouping by concept.\n        order=torch.argsort(agg,descending=True,stable=True)\n        concept=concept[order]; item=item[order]\n        order=torch.argsort(concept,stable=True)\n        concept=concept[order]; item=item[order]\n\n        counts=torch.bincount(concept,minlength=self.n_concepts)\n        starts=torch.cumsum(counts,0)-counts\n        local=torch.arange(concept.numel())-torch.repeat_interleave(starts,counts)\n        keep=(local<self.terminal_degree)&(item!=0)\n\n        kc=concept[keep].to(self.terminal_items.device)\n        ki=item[keep].to(self.terminal_items.device,dtype=torch.int32)\n        kr=local[keep].to(self.terminal_items.device)\n        self.terminal_items[kc,kr]=ki\n\n        touched=int(torch.unique(concept).numel())\n        self.begin_terminal_epoch()\n        return touched\n\n    @torch.inference_mode()\n    def full_scores(self,seq,lengths):\n        # Returns a dense-shaped score tensor for the common evaluator, but scores\n        # only items reachable through the sparse terminal support.\n        H,I,_=self.encode_with_states(seq)\n        rows=torch.arange(seq.size(0),device=seq.device)\n        ix=lengths-1\n        h=H[rows,ix]; ids=I[rows,ix]\n        cand=self.terminal_items[ids].long().reshape(seq.size(0),-1)\n        cs=(self.item.weight[cand]*h[:,None,:]).sum(-1).float()\n        table=torch.full((seq.size(0),self.n_items+1),-1e9,device=seq.device,dtype=torch.float32)\n        table=table.scatter_reduce(1,cand,cs,reduce="amax",include_self=True)\n        table[:,0]=-1e9\n        return table\n\n# %%\n# ============================================================\n# 13. MODEL FACTORY / LOSS MODES\n# ============================================================\n\ndef build_model(name,n_items,max_len,dataset_name=None):\n    d=CFG.d_model\n    if name=="GRU4Rec": return GRU4RecModel(n_items,max_len,d,CFG.layers,CFG.dropout)\n    if name=="SASRec" or name=="SASRec+SS": return SASRecModel(n_items,max_len,d,CFG.layers,CFG.heads,CFG.ffn_dim,CFG.dropout,False)\n    if name=="eSASRec": return SASRecModel(n_items,max_len,d,CFG.layers,CFG.heads,CFG.ffn_dim,CFG.dropout,True)\n    if name in ("HSTU-core","HSTU-large"):\n        a=hstu_architecture(dataset_name or "beauty", large=(name=="HSTU-large"))\n        return HSTUModel(\n            n_items,max_len,\n            d=a["d"],layers=a["layers"],heads=a["heads"],\n            dqk=a["dqk"],dv=a["dv"],dropout=a["dropout"],\n            l2_eps=CFG.hstu_l2_eps,\n        )\n    if name=="FMLP-Rec": return FMLPRecModel(n_items,max_len,d,CFG.layers,CFG.ffn_dim,CFG.dropout)\n    if name=="Mamba4Rec": return Mamba4RecModel(n_items,max_len,d,CFG.mamba_layers,CFG.dropout,CFG.mamba_d_state,CFG.mamba_d_conv,CFG.mamba_expand)\n    if name=="BERT4Rec": return BERT4RecModel(n_items,max_len,d,CFG.layers,CFG.heads,CFG.ffn_dim,CFG.dropout)\n    if name=="SparseWalker": return SparseWalkerModel(n_items,max_len,d,CFG.layers,CFG.concept_side,CFG.concept_dim,CFG.active_concepts,CFG.fresh_top_side,CFG.graph_degree,CFG.fresh_weight)\n    if name=="SparseWalker-E2E": return SparseWalkerE2EModel(n_items,max_len,d,CFG.layers,CFG.concept_side,CFG.concept_dim,CFG.active_concepts,CFG.fresh_top_side,CFG.graph_degree,CFG.fresh_weight,CFG.e2e_terminal_degree)\n    raise KeyError(name)\n\n\ndef loss_mode_for(name, n_items, large_catalog=False, dataset_name=None, max_len=50):\n    # HSTU public configurations use sampled softmax + normalized embeddings.\n    if name in ("HSTU-core","HSTU-large"): return "hstu_ss"\n    # eSASRec\'s canonical recipe and SASRec+SS explicitly use sampled softmax.\n    if name == "SparseWalker-E2E": return "e2e_sparse"\n    if name in ("SASRec+SS", "eSASRec"): return "ss"\n    # For the long-sequence / large-corpus suites, use one shared sampled-softmax\n    # objective for all autoregressive bodies. This keeps the comparison feasible\n    # on a 40 GB Colab GPU and mirrors the public HSTU benchmark recipe.\n    if dataset_name in ("ml1m", "ml20m", "books") or large_catalog or n_items > 50_000:\n        return "ss"\n    # Small Amazon 5-core suites retain the all-position full-catalog CE protocol\n    # used in our Beauty reconciliation experiment.\n    return "full"\n\n# %%\n# ============================================================\n# 14. EVALUATION\n# ============================================================\n\ndef make_eval_batch(prefixes, indices, max_len):\n    seqs=[prefixes[i][-max_len:] for i in indices]; lens=torch.tensor([len(s) for s in seqs],dtype=torch.long); L=int(lens.max())\n    x=torch.zeros(len(seqs),L,dtype=torch.long)\n    for r,s in enumerate(seqs): x[r,:len(s)]=torch.tensor(s)\n    return x,lens\n\n\ndef head_items_from_freq(freq,n_items,frac=.2):\n    counts=np.zeros(n_items+1,dtype=np.float64)\n    for i,c in freq.items():\n        if int(i)<=n_items: counts[int(i)]=c\n    k=max(1,int(n_items*frac)); ids=np.argsort(-counts[1:])[:k]+1; return set(ids.tolist())\n\n@torch.inference_mode()\ndef evaluate_full(model, prefixes, targets, n_items, max_len, item_freq, batch_size=None, save_top=False):\n    model.eval(); topks=sorted(set(k for k in CFG.topks if k<=n_items)); maxk=max(topks)\n    # cap score matrix around ~32M entries\n    dynamic=max(16,min(batch_size or CFG.eval_batch_size,int(32_000_000/max(1,n_items))))\n    head=head_items_from_freq(item_freq,n_items,CFG.head_fraction)\n    sums={f"HR@{k}":0. for k in topks}; sums.update({f"NDCG@{k}":0. for k in topks})\n    mrr10=0.; coverage10=set(); coverage20=set(); head_hit=head_n=tail_hit=tail_n=0.; rec_pop_sum=0.; rec_pop_n=0\n    tops_saved=[]; total=len(targets)\n    freq_arr=np.zeros(n_items+1,dtype=np.float64)\n    for i,c in item_freq.items():\n        if int(i)<=n_items: freq_arr[int(i)]=c\n    logpop=np.log1p(freq_arr)\n    for start in range(0,total,dynamic):\n        end=min(total,start+dynamic); idxs=list(range(start,end)); seq,lens=make_eval_batch(prefixes,idxs,max_len)\n        seq=seq.to(DEVICE,non_blocking=True); lens=lens.to(DEVICE,non_blocking=True); tgt=torch.tensor(targets[start:end],device=DEVICE)\n        with torch.autocast("cuda",dtype=AMP_DTYPE,enabled=torch.cuda.is_available()): scores=model.full_scores(seq,lens)\n        scores=scores.float()\n        for r,i in enumerate(idxs):\n            seen=set(prefixes[i]); truth=int(tgt[r]); seen.discard(truth)\n            if seen:\n                ids=torch.tensor(list(seen),device=DEVICE); scores[r,ids]=-1e20\n        top=scores.topk(maxk,dim=-1).indices; top_np=top.cpu().numpy(); tgt_np=tgt.cpu().numpy()\n        if save_top: tops_saved.append(top_np[:,:max(20,maxk)])\n        coverage10.update(top_np[:,:min(10,maxk)].reshape(-1).tolist())\n        coverage20.update(top_np[:,:min(20,maxk)].reshape(-1).tolist())\n        rec_pop_sum += float(logpop[top_np[:,:min(10,maxk)]].sum()); rec_pop_n += top_np[:,:min(10,maxk)].size\n        for r,truth in enumerate(tgt_np):\n            pos=np.where(top_np[r]==truth)[0]; rank=int(pos[0])+1 if len(pos) else None\n            for k in topks:\n                if rank is not None and rank<=k:\n                    sums[f"HR@{k}"]+=1; sums[f"NDCG@{k}"]+=1/math.log2(rank+1)\n            if rank is not None and rank<=10: mrr10 += 1/rank\n            if int(truth) in head:\n                head_n+=1; head_hit += int(rank is not None and rank<=10)\n            else:\n                tail_n+=1; tail_hit += int(rank is not None and rank<=10)\n    out={k:v/total for k,v in sums.items()}; out.update({\n        "MRR@10":mrr10/total,"Coverage@10":len(coverage10)/n_items,"Coverage@20":len(coverage20)/n_items,\n        "HeadHR@10":head_hit/max(1,head_n),"TailHR@10":tail_hit/max(1,tail_n),"MeanLogPop@10":rec_pop_sum/max(1,rec_pop_n),\n    })\n    if save_top: out["top_items"]=np.concatenate(tops_saved,0)\n    return out\n\n\ndef deterministic_sampled_candidates(prefix,target,n_items,n_negs,user_idx,seed=12345):\n    rng=random.Random(seed*1_000_003+user_idx); seen=set(prefix); seen.add(target); neg=[]\n    while len(neg)<n_negs:\n        x=rng.randint(1,n_items)\n        if x not in seen: seen.add(x); neg.append(x)\n    return [target]+neg\n\n@torch.inference_mode()\ndef evaluate_sampled(model,prefixes,targets,n_items,max_len,n_negs=100):\n    model.eval(); hits=ndcg=mrr=0.; total=len(targets); bs=min(CFG.eval_batch_size,1024)\n    for start in range(0,total,bs):\n        end=min(total,start+bs); idxs=list(range(start,end)); seq,lens=make_eval_batch(prefixes,idxs,max_len)\n        cand=np.asarray([deterministic_sampled_candidates(prefixes[i],targets[i],n_items,n_negs,i) for i in idxs],dtype=np.int64)\n        seq=seq.to(DEVICE); lens=lens.to(DEVICE); c=torch.tensor(cand,device=DEVICE)\n        with torch.autocast("cuda",dtype=AMP_DTYPE,enabled=torch.cuda.is_available()):\n            scores=model.full_scores(seq,lens).gather(1,c)\n        # Deterministic tie break: higher score first; on equal score,\n        # smaller item ID ranks first. This prevents the positive in column 0\n        # from automatically winning every tie (which made MostPop HR@10 ~1).\n        pos_score=scores[:,0:1]\n        pos_id=c[:,0:1]\n        rank=((scores>pos_score) | ((scores==pos_score) & (c<pos_id))).sum(1).cpu().numpy()\n        hits += float((rank<10).sum())\n        ndcg += float(sum(1/math.log2(int(r)+2) for r in rank if r<10))\n        mrr += float(sum(1/(int(r)+1) for r in rank if r<10))\n    return {"sampled_HR@10":hits/total,"sampled_NDCG@10":ndcg/total,"sampled_MRR@10":mrr/total}\n\n# %%\n# ============================================================\n# 15. MOSTPOP\n# ============================================================\n\nclass MostPopModel(nn.Module):\n    def __init__(self,n_items,freq):\n        super().__init__(); self.n_items=n_items; x=torch.zeros(n_items+1)\n        for i,c in freq.items():\n            if int(i)<=n_items: x[int(i)]=float(c)\n        x[0]=-1e9; self.register_buffer("scores",x)\n    def full_scores(self,seq,lengths): return self.scores[None].expand(seq.size(0),-1)\n\n# %%\n# ============================================================\n# 16. TRAINING\n# ============================================================\n\ndef training_recipe(name,dataset_name):\n    """Return optimizer/schedule settings.\n\n    HSTU follows Meta\'s public recipe:\n      AdamW beta=(.9,.98), lr=1e-3 constant, wd=0, no warmup.\n      Amazon L=50 uses 512 sampled negatives; MovieLens uses 128.\n    """\n    if name in ("HSTU-core","HSTU-large"):\n        amazon = dataset_name not in ("ml1m","ml20m")\n        return {\n            "lr":1e-3,\n            "optimizer":"adamw",\n            "betas":(0.9,0.98),\n            "weight_decay":0.0,\n            "schedule":"constant",\n            "batch_size":CFG.hstu_batch_size,\n            "max_epochs":CFG.hstu_amazon_max_epochs if amazon else CFG.hstu_max_epochs,\n            "patience":CFG.hstu_patience,\n            "n_negs":CFG.hstu_amazon_negatives if amazon else CFG.hstu_movielens_negatives,\n        }\n    return {\n        "lr":CFG.lr_by_model[name],\n        "optimizer":"adam",\n        "betas":(0.9,0.999),\n        "weight_decay":CFG.weight_decay,\n        "schedule":"cosine",\n        "batch_size":CFG.batch_size,\n        "max_epochs":CFG.max_epochs,\n        "patience":CFG.patience,\n        "n_negs":CFG.n_negs_train,\n    }\n\n\ndef train_model(name, model, train_ds, val_prefix, val_target, n_items, max_len, item_freq, outdir, seed, large_catalog=False, dataset_name=None):\n    version = "hstu_v3" if name in ("HSTU-core","HSTU-large") else "v2"\n    ckpt=outdir/f"{safe_name(name)}_{version}_seed{seed}.pt"\n    meta=outdir/f"{safe_name(name)}_{version}_seed{seed}_train.json"\n    if CFG.reuse and ckpt.exists() and meta.exists():\n        model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True)); return json.load(open(meta))\n    recipe=training_recipe(name,dataset_name)\n    peak=recipe["lr"]\n    if recipe["optimizer"]=="adamw":\n        opt=torch.optim.AdamW(\n            model.parameters(),lr=peak,betas=recipe["betas"],\n            weight_decay=recipe["weight_decay"],\n        )\n    else:\n        opt=torch.optim.Adam(\n            model.parameters(),lr=peak,betas=recipe["betas"],\n            weight_decay=recipe["weight_decay"],\n        )\n    best=-1.; best_epoch=-1; bad=0; history=[]; t_all=time.perf_counter(); mode=loss_mode_for(name,n_items,large_catalog,dataset_name,max_len)\n    for epoch in range(1,recipe["max_epochs"]+1):\n        lr = peak if recipe["schedule"]=="constant" else cosine_lr(\n            epoch,recipe["max_epochs"],peak,CFG.min_lr,CFG.warmup_epochs\n        )\n        for g in opt.param_groups:g["lr"]=lr\n        model.train(); ls=0.; nb=0; t0=time.perf_counter()\n        if name=="SparseWalker-E2E": model.begin_terminal_epoch()\n        for tokens,lengths in make_train_loader(train_ds,epoch,recipe[\'batch_size\']):\n            tokens=tokens.to(DEVICE,non_blocking=True); lengths=lengths.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True)\n            with torch.autocast("cuda",dtype=AMP_DTYPE,enabled=torch.cuda.is_available()):\n                if name=="BERT4Rec": loss=model.masked_loss(tokens,lengths,CFG.bert_mask_prob,mode,CFG.n_negs_train)\n                elif name=="SparseWalker-E2E": loss=model.sparse_training_loss(tokens,CFG.e2e_evidence_per_batch,CFG.e2e_loss_chunk)\n                else: loss=ar_training_loss(model,tokens,lengths,mode,recipe[\'n_negs\'])\n            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),CFG.grad_clip); opt.step(); ls+=float(loss.detach()); nb+=1\n        pursued=0\n        if name in ("SparseWalker","SparseWalker-E2E") and epoch>=CFG.pursuit_start and (epoch-CFG.pursuit_start)%CFG.pursuit_every==0:\n            pursued=model.graph.pursue(opt,CFG.pursuit_refresh)\n        terminal_refreshed=0\n        if name=="SparseWalker-E2E" and epoch>=CFG.e2e_terminal_refresh_start and (epoch-CFG.e2e_terminal_refresh_start)%CFG.e2e_terminal_refresh_every==0:\n            terminal_refreshed=model.refresh_terminal()\n        val=evaluate_full(model,val_prefix,val_target,n_items,max_len,item_freq)\n        row={"epoch":epoch,"lr":lr,"loss":ls/max(1,nb),"val_NDCG@10":val.get("NDCG@10",0),"val_HR@10":val.get("HR@10",0),"seconds":time.perf_counter()-t0,"pursued":pursued,"terminal_refreshed_concepts":terminal_refreshed}\n        history.append(row); print(name,row)\n        score=val.get("NDCG@10",0)\n        if score>best+1e-8:\n            best=score; best_epoch=epoch; bad=0; torch.save(model.state_dict(),ckpt)\n        else: bad+=1\n        if bad>=recipe[\'patience\']: break\n    model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True))\n    info={"best_val_NDCG@10":best,"best_epoch":best_epoch,"train_seconds":time.perf_counter()-t_all,"loss_mode":mode,"training_recipe":recipe,"history":history}\n    json.dump(info,open(meta,"w"),indent=2); return info\n\n# %%\n# ============================================================\n# 17. LATENCY BENCHMARK FOR DENSE MODELS\n# ============================================================\n\ndef benchmark_cuda(fn,warmup=None,repeats=None):\n    if not torch.cuda.is_available(): return {"median_ms":float("nan"),"p95_ms":float("nan")}\n    warmup=CFG.latency_warmup if warmup is None else warmup; repeats=CFG.latency_repeats if repeats is None else repeats\n    for _ in range(warmup):fn()\n    torch.cuda.synchronize(); vals=[]\n    for _ in range(repeats):\n        a=torch.cuda.Event(enable_timing=True); b=torch.cuda.Event(enable_timing=True); a.record(); fn(); b.record(); torch.cuda.synchronize(); vals.append(a.elapsed_time(b))\n    return {"median_ms":float(np.median(vals)),"p95_ms":float(np.percentile(vals,95))}\n\n@torch.inference_mode()\ndef benchmark_dense_request(model,prefixes,max_len):\n    # batch=1 complete request: sequence encode + full-catalog score + top10\n    s=prefixes[np.argmax([len(x) for x in prefixes])][-max_len:]; x=torch.tensor(s,device=DEVICE)[None]; l=torch.tensor([len(s)],device=DEVICE)\n    def fn(): model.full_scores(x,l).topk(10,-1)\n    return benchmark_cuda(fn)\n\n# %%\n# ============================================================\n# 18. WALKER TERMINAL COMPILER (MEMORY-BOUNDED)\n# ============================================================\n\n@torch.inference_mode()\ndef collect_walker_events(model: SparseWalkerModel, train_seqs, max_len, max_events, seed):\n    # Sample up to max_events autoregressive positions across users. This keeps\n    # ML-20M/Books compilation bounded while Beauty still uses essentially all.\n    rng=np.random.default_rng(seed); records=[]\n    for u,s in enumerate(train_seqs):\n        for t in range(1,len(s)):\n            records.append((u,t))\n            if len(records)>=max_events*4: break\n        if len(records)>=max_events*4: break\n    # If corpus order gave too many early-user positions, reservoir-like subsample.\n    if len(records)>max_events:\n        pick=rng.choice(len(records),size=max_events,replace=False); records=[records[i] for i in pick]\n    # For very large corpora above shortcut only sees early users; replace with randomized user sampling.\n    if sum(max(0,len(s)-1) for s in train_seqs)>max_events*4:\n        records=[]; user_order=rng.permutation(len(train_seqs))\n        for u in user_order:\n            s=train_seqs[u]\n            if len(s)<2:continue\n            ts=np.arange(1,len(s)); rng.shuffle(ts)\n            for t in ts[:min(len(ts),8)]:\n                records.append((int(u),int(t)))\n                if len(records)>=max_events:break\n            if len(records)>=max_events:break\n    ids_all=[]; mass_all=[]; hidden_all=[]; target_all=[]\n    bs=512\n    for st in range(0,len(records),bs):\n        chunk=records[st:st+bs]; seqs=[]; lens=[]; tgt=[]\n        for u,t in chunk:\n            hist=train_seqs[u][max(0,t-max_len):t]; seqs.append(hist); lens.append(len(hist)); tgt.append(train_seqs[u][t])\n        L=max(lens); x=torch.zeros(len(chunk),L,dtype=torch.long)\n        for r,s in enumerate(seqs):x[r,:len(s)]=torch.tensor(s)\n        x=x.to(DEVICE); lengths=torch.tensor(lens,device=DEVICE)\n        H,I,M=model.encode_with_states(x); rows=torch.arange(len(chunk),device=DEVICE); ix=lengths-1\n        ids_all.append(I[rows,ix].to(torch.int32).cpu()); mass_all.append(M[rows,ix].float().cpu()); hidden_all.append(H[rows,ix].to(torch.bfloat16).cpu()); target_all.append(torch.tensor(tgt))\n    return {"state_ids":torch.cat(ids_all),"state_mass":torch.cat(mass_all),"hidden":torch.cat(hidden_all),"target":torch.cat(target_all)}\n\n@torch.inference_mode()\ndef teacher_topk(model,hidden,k):\n    emb=model.item.weight[:model.n_items+1].detach().to(DEVICE,dtype=AMP_DTYPE); out=[]\n    for st in range(0,hidden.size(0),2048):\n        h=hidden[st:st+2048].to(DEVICE,dtype=AMP_DTYPE); z=h@emb.T; z[:,0]=-1e9; out.append(z.topk(k,-1).indices.cpu())\n    return torch.cat(out)\n\n\ndef compile_terminal_support(model, train_seqs, item_freq, n_items, max_len, outdir, seed):\n    path=outdir/f"terminal_support_seed{seed}.pt"\n    if CFG.reuse and path.exists(): return torch.load(path,weights_only=False)\n    feat=collect_walker_events(model,train_seqs,max_len,CFG.compiler_max_events,seed)\n    feat["teacher"]=teacher_topk(model,feat["hidden"],CFG.teacher_topk)\n    concept=feat["state_ids"].long(); mass=feat["state_mass"].float(); teacher=feat["teacher"][:,:CFG.compiler_teacher_k].long(); target=feat["target"].long()\n    K=concept.size(1); T=teacher.size(1); rank_weight=1/torch.log2(torch.arange(T,dtype=torch.float32)+2)\n    partial_pairs=[]; partial_weights=[]\n    for st in range(0,concept.size(0),CFG.compiler_chunk_events):\n        en=min(concept.size(0),st+CFG.compiler_chunk_events); c=concept[st:en]; m=mass[st:en]; ti=teacher[st:en]\n        pc=(c[:,:,None].expand(-1,K,T)*(n_items+1)+ti[:,None,:].expand(-1,K,-1)).reshape(-1)\n        pw=(m[:,:,None]*rank_weight[None,None,:]).reshape(-1)\n        pt=(c*(n_items+1)+target[st:en,None]).reshape(-1); tw=(CFG.target_bonus*m).reshape(-1)\n        p=torch.cat([pc,pt]); w=torch.cat([pw,tw]); order=torch.argsort(p); p=p[order]; w=w[order]\n        up,inv=torch.unique_consecutive(p,return_inverse=True); aw=torch.zeros(up.numel(),dtype=torch.float32); aw.scatter_add_(0,inv,w)\n        partial_pairs.append(up); partial_weights.append(aw)\n    pairs=torch.cat(partial_pairs); weights=torch.cat(partial_weights); order=torch.argsort(pairs); pairs=pairs[order]; weights=weights[order]\n    up,inv=torch.unique_consecutive(pairs,return_inverse=True); agg=torch.zeros(up.numel(),dtype=torch.float32); agg.scatter_add_(0,inv,weights)\n    cid=up//(n_items+1); iid=up%(n_items+1); total=torch.zeros(model.n_concepts,dtype=torch.float32); total.scatter_add_(0,cid,agg)\n    score=torch.log(agg+1e-12)-torch.log(total[cid]+1e-12)\n    # Sort by concept then descending score via stable two-stage sort.\n    o=torch.argsort(score,descending=True,stable=True); cid,iid,score=cid[o],iid[o],score[o]; o=torch.argsort(cid,stable=True); cid,iid,score=cid[o],iid[o],score[o]\n    counts=torch.bincount(cid,minlength=model.n_concepts); starts=torch.cumsum(counts,0)-counts; local=torch.arange(cid.numel())-torch.repeat_interleave(starts,counts); keep=local<CFG.terminal_max_degree\n    pop=torch.zeros(n_items+1)\n    for i,c in item_freq.items():\n        if int(i)<=n_items:pop[int(i)]=float(c)\n    popular=torch.argsort(pop,descending=True); popular=popular[popular!=0][:CFG.terminal_max_degree]\n    table=popular[None].expand(model.n_concepts,-1).clone().to(torch.int32); stable_scores=torch.full((model.n_concepts,CFG.terminal_max_degree),-30.,dtype=torch.float32)\n    kc,ki,ks,kr=cid[keep],iid[keep],score[keep],local[keep]; table[kc,kr]=ki.to(torch.int32); stable_scores[kc,kr]=ks\n    out={"item":table,"score":stable_scores,"observed_counts":counts,"compiler_events":int(concept.size(0)),"unique_relations":int(up.numel())}\n    torch.save(out,path); return out\n\n@torch.inference_mode()\ndef final_walker_features(model,prefixes,targets,max_len):\n    ids_all=[];mass_all=[];hidden_all=[];teacher_all=[];bs=min(CFG.eval_batch_size,1024)\n    for st in range(0,len(prefixes),bs):\n        en=min(len(prefixes),st+bs); idx=list(range(st,en)); x,l=make_eval_batch(prefixes,idx,max_len); x=x.to(DEVICE);l=l.to(DEVICE)\n        H,I,M=model.encode_with_states(x); rows=torch.arange(x.size(0),device=DEVICE); ix=l-1; h=H[rows,ix]\n        ids_all.append(I[rows,ix].to(torch.int32).cpu()); mass_all.append(M[rows,ix].float().cpu()); hidden_all.append(h.to(torch.bfloat16).cpu()); teacher_all.append(teacher_topk(model,h,10))\n    return {"ids":torch.cat(ids_all),"mass":torch.cat(mass_all),"hidden":torch.cat(hidden_all),"teacher":torch.cat(teacher_all),"target":torch.tensor(targets)}\n\n@torch.inference_mode()\ndef evaluate_terminal(model,support,features,prefixes,n_items,degree,item_freq):\n    supp=support["item"][:,:degree].to(DEVICE); emb=model.item.weight.detach().to(DEVICE,dtype=AMP_DTYPE)\n    total=features["target"].size(0); cand_hit=teacher_rec=hr=ndcg=mrr=0.; cov=set(); bs=min(CFG.eval_batch_size,512)\n    for st in range(0,total,bs):\n        en=min(total,st+bs); ids=features["ids"][st:en].to(DEVICE).long(); h=features["hidden"][st:en].to(DEVICE,dtype=AMP_DTYPE); tgt=features["target"][st:en].to(DEVICE); tea=features["teacher"][st:en].to(DEVICE)\n        B=ids.size(0); cand=supp[ids].reshape(B,-1); cand_hit+=int((cand==tgt[:,None]).any(-1).sum())\n        teacher_rec+=float((tea[:,:,None]==cand[:,None,:]).any(-1).float().mean(-1).sum())\n        score=(emb[cand]*h[:,None,:]).sum(-1).float(); table=torch.full((B,n_items+1),-1e20,device=DEVICE); table.scatter_reduce_(1,cand,score,reduce="amax",include_self=True)\n        for r in range(B):\n            seen=set(prefixes[st+r]); truth=int(tgt[r]); seen.discard(truth)\n            if seen:table[r,torch.tensor(list(seen),device=DEVICE)]=-1e20\n        top=table.topk(10,-1).indices; cov.update(top.flatten().cpu().tolist()); match=top==tgt[:,None]\n        for r in range(B):\n            p=match[r].nonzero(as_tuple=False)\n            if p.numel():\n                rank=int(p[0,0])+1;hr+=1;ndcg+=1/math.log2(rank+1);mrr+=1/rank\n    return {"degree":degree,"candidates":CFG.active_concepts*degree,"candidate_recall":cand_hit/total,"teacher_top10_recall":teacher_rec/total,\n            "HR@10":hr/total,"NDCG@10":ndcg/total,"MRR@10":mrr/total,"Coverage@10":len(cov)/n_items}\n\n# %%\n# ============================================================\n# 19. TRITON FUSED WALKER SERVING (D=64/H=16/K=8/DEG=4)\n# ============================================================\n\nif HAS_TRITON:\n    @triton.jit\n    def route_kernel(ITEM_ID,ITEM_EMBEDDING,QUERY_WEIGHT,LEFT_KEYS,RIGHT_KEYS,FRESH_IDS,FRESH_MASS,Q_CONTEXT,D:tl.constexpr,H:tl.constexpr,SIDE:tl.constexpr,INPUT_SCALE:tl.constexpr):\n        d=tl.arange(0,64);h=tl.arange(0,16);s=tl.arange(0,256);item=tl.load(ITEM_ID).to(tl.int32)\n        x=tl.load(ITEM_EMBEDDING+item*D+d,mask=d<D,other=0.).to(tl.float32)*INPUT_SCALE\n        wl=tl.load(QUERY_WEIGHT+h[:,None]*D+d[None,:],mask=(h[:,None]<H)&(d[None,:]<D),other=0.).to(tl.float32)\n        wr=tl.load(QUERY_WEIGHT+H*D+h[:,None]*D+d[None,:],mask=(h[:,None]<H)&(d[None,:]<D),other=0.).to(tl.float32)\n        wc=tl.load(QUERY_WEIGHT+2*H*D+h[:,None]*D+d[None,:],mask=(h[:,None]<H)&(d[None,:]<D),other=0.).to(tl.float32)\n        ql=tl.sum(wl*x[None,:],axis=1);qr=tl.sum(wr*x[None,:],axis=1);qc=tl.sum(wc*x[None,:],axis=1)\n        ql/=tl.sqrt(tl.sum(ql*ql,axis=0)+1e-8);qr/=tl.sqrt(tl.sum(qr*qr,axis=0)+1e-8);qc/=tl.sqrt(tl.sum(qc*qc,axis=0)+1e-8);tl.store(Q_CONTEXT+h,qc,mask=h<H)\n        lk=tl.load(LEFT_KEYS+s[:,None]*H+h[None,:],mask=(s[:,None]<SIDE)&(h[None,:]<H),other=0.).to(tl.float32);rk=tl.load(RIGHT_KEYS+s[:,None]*H+h[None,:],mask=(s[:,None]<SIDE)&(h[None,:]<H),other=0.).to(tl.float32)\n        ls=tl.sum(lk*ql[None,:],axis=1);rs=tl.sum(rk*qr[None,:],axis=1)\n        li0=tl.argmax(ls,axis=0);lv0=tl.max(ls,axis=0);ls2=tl.where(s==li0,-1e20,ls);li1=tl.argmax(ls2,axis=0);lv1=tl.max(ls2,axis=0)\n        ri0=tl.argmax(rs,axis=0);rv0=tl.max(rs,axis=0);rs2=tl.where(s==ri0,-1e20,rs);ri1=tl.argmax(rs2,axis=0);rv1=tl.max(rs2,axis=0)\n        f=tl.arange(0,4);lid=tl.where(f<2,li0,li1);lv=tl.where(f<2,lv0,lv1);rid=tl.where((f%2)==0,ri0,ri1);rv=tl.where((f%2)==0,rv0,rv1)\n        logits=lv+rv;p=tl.exp(logits-tl.max(logits,axis=0));p/=tl.sum(p,axis=0)+1e-8;tl.store(FRESH_IDS+f,(lid*SIDE+rid).to(tl.int32));tl.store(FRESH_MASS+f,p)\n\n    @triton.jit\n    def walk_kernel(STATE_IDS,STATE_MASS,FRESH_IDS,FRESH_MASS,QUERY,DESTINATION,EDGE,DEST_KEY,OUT_IDS,OUT_MASS,K:tl.constexpr,DEGREE:tl.constexpr,H:tl.constexpr):\n        k=tl.arange(0,K);ms=tl.arange(0,16);path=tl.arange(0,32);h=tl.arange(0,16)\n        ids=tl.load(STATE_IDS+k).to(tl.int32);mass=tl.load(STATE_MASS+k).to(tl.float32);f=tl.arange(0,4);fi=tl.load(FRESH_IDS+f).to(tl.int32);fm=tl.load(FRESH_MASS+f).to(tl.float32);q=tl.load(QUERY+h,mask=h<H,other=0.).to(tl.float32)\n        for _ in range(2):\n            oi=tl.minimum(ms,K-1);ni=tl.maximum(tl.minimum(ms-K,3),0);oid=tl.gather(ids,oi,axis=0);om=tl.gather(mass,oi,axis=0);nid=tl.gather(fi,ni,axis=0);nm=tl.gather(fm,ni,axis=0)\n            mid=tl.where(ms<K,oid,tl.where(ms<K+4,nid,0));mm=tl.where(ms<K,.75*om,tl.where(ms<K+4,.25*nm,-1e20));chosen_id=tl.full((K,),0,tl.int32);chosen_mass=tl.zeros((K,),tl.float32);scores=mm\n            for j in range(K):\n                ix=tl.argmax(scores,axis=0);v=tl.max(scores,axis=0);cid=tl.sum(tl.where(ms==ix,mid,0),axis=0);chosen_id=tl.where(k==j,cid,chosen_id);chosen_mass=tl.where(k==j,v,chosen_mass);scores=tl.where(ms==ix,-1e20,scores)\n            chosen_mass/=tl.sum(chosen_mass,axis=0)+1e-8;src=path//DEGREE;es=path%DEGREE;sid=tl.gather(chosen_id,src,axis=0);gi=sid*DEGREE+es;dest=tl.load(DESTINATION+gi).to(tl.int32);static=tl.load(EDGE+gi).to(tl.float32)\n            key=tl.load(DEST_KEY+dest[:,None]*H+h[None,:],mask=h[None,:]<H,other=0.).to(tl.float32);ctx=tl.sum(key*q[None,:],axis=1);score=tl.reshape(static+ctx,(K,DEGREE));rm=tl.max(score,axis=1);prob=tl.exp(score-rm[:,None]);prob/=tl.sum(prob,axis=1)[:,None]+1e-8;pm=tl.reshape(prob*chosen_mass[:,None],(32,))\n            nids=tl.full((K,),0,tl.int32);nmass=tl.zeros((K,),tl.float32);scores=pm\n            for j in range(K):\n                ix=tl.argmax(scores,axis=0);v=tl.max(scores,axis=0);did=tl.sum(tl.where(path==ix,dest,0),axis=0);nids=tl.where(k==j,did,nids);nmass=tl.where(k==j,v,nmass);scores=tl.where(path==ix,-1e20,scores)\n            ids=nids;mass=nmass/(tl.sum(nmass,axis=0)+1e-8)\n        tl.store(OUT_IDS+k,ids);tl.store(OUT_MASS+k,mass)\n\n    @triton.jit\n    def readout_kernel(ITEM_ID,STATE_IDS,STATE_MASS,CONCEPT_VALUES,MESSAGE_WEIGHT,NORM_WEIGHT,NORM_BIAS,ITEM_EMBEDDING,HIDDEN,K:tl.constexpr,D:tl.constexpr,INPUT_SCALE:tl.constexpr):\n        k=tl.arange(0,K);d=tl.arange(0,64);sid=tl.load(STATE_IDS+k).to(tl.int32);sm=tl.load(STATE_MASS+k).to(tl.float32)\n        val=tl.load(CONCEPT_VALUES+sid[:,None]*D+d[None,:],mask=d[None,:]<D,other=0.).to(tl.float32);msg=tl.sum(val*sm[:,None],axis=0)\n        w=tl.load(MESSAGE_WEIGHT+d[:,None]*D+d[None,:],mask=(d[:,None]<D)&(d[None,:]<D),other=0.).to(tl.float32);proj=tl.sum(w*msg[None,:],axis=1);item=tl.load(ITEM_ID).to(tl.int32);iv=tl.load(ITEM_EMBEDDING+item*D+d,mask=d<D,other=0.).to(tl.float32);res=INPUT_SCALE*iv+proj\n        mean=tl.sum(res,axis=0)/D;c=res-mean;var=tl.sum(c*c,axis=0)/D;nw=tl.load(NORM_WEIGHT+d,mask=d<D,other=1.).to(tl.float32);nb=tl.load(NORM_BIAS+d,mask=d<D,other=0.).to(tl.float32);out=c*tl.rsqrt(var+1e-5)*nw+nb;tl.store(HIDDEN+d,out,mask=d<D)\n\n    @triton.jit\n    def terminal_block_kernel(HIDDEN,STATE_IDS,SUPPORT_ITEM,ITEM_EMBEDDING,BLOCK_ITEMS,BLOCK_SCORES,DEGREE:tl.constexpr,D:tl.constexpr,TOTAL_CANDIDATES:tl.constexpr,KEEP:tl.constexpr):\n        bid=tl.program_id(0);lane=tl.arange(0,128);d=tl.arange(0,64);path=bid*128+lane;valid=path<TOTAL_CANDIDATES;src=path//DEGREE;es=path%DEGREE;sid=tl.load(STATE_IDS+src,mask=valid,other=0).to(tl.int32);item=tl.load(SUPPORT_ITEM+sid*DEGREE+es,mask=valid,other=0).to(tl.int32);h=tl.load(HIDDEN+d,mask=d<D,other=0.).to(tl.float32);e=tl.load(ITEM_EMBEDDING+item[:,None]*D+d[None,:],mask=valid[:,None]&(d[None,:]<D),other=0.).to(tl.float32);score=tl.sum(e*h[None,:],axis=1);score=tl.where(valid&(item!=0),score,-1e20)\n        for j in range(KEEP):\n            ix=tl.argmax(score,axis=0);v=tl.max(score,axis=0);ci=tl.sum(tl.where(lane==ix,item,0),axis=0);tl.store(BLOCK_ITEMS+bid*KEEP+j,ci);tl.store(BLOCK_SCORES+bid*KEEP+j,v);score=tl.where(item==ci,-1e20,score)\n\n    @triton.jit\n    def terminal_merge_kernel(BLOCK_ITEMS,BLOCK_SCORES,OUTPUT,COUNT:tl.constexpr,BLOCK:tl.constexpr):\n        lane=tl.arange(0,BLOCK);valid=lane<COUNT;item=tl.load(BLOCK_ITEMS+lane,mask=valid,other=0).to(tl.int32);score=tl.load(BLOCK_SCORES+lane,mask=valid,other=-1e20).to(tl.float32);score=tl.where(valid,score,-1e20)\n        for j in range(10):\n            ix=tl.argmax(score,axis=0);ci=tl.sum(tl.where(lane==ix,item,0),axis=0);tl.store(OUTPUT+j,ci);score=tl.where(item==ci,-1e20,score)\n\nclass FastWalkerArtifacts:\n    def __init__(self,model):\n        assert HAS_TRITON and model.d_model==64 and model.h==16 and model.side==256 and model.active==8 and model.degree==4 and model.layers_n==2\n        self.item=model.item.weight.detach().to(DEVICE,dtype=AMP_DTYPE).contiguous(); self.query_weight=torch.stack([model.router.left_q.weight,model.router.right_q.weight,model.graph.context_q.weight]).detach().to(DEVICE,dtype=AMP_DTYPE).contiguous()\n        rs=torch.exp(model.router.scale.detach()); self.left_keys=(F.normalize(model.space.left_router.detach().float(),dim=-1)*rs).to(DEVICE,dtype=AMP_DTYPE).contiguous(); self.right_keys=(F.normalize(model.space.right_router.detach().float(),dim=-1)*rs).to(DEVICE,dtype=AMP_DTYPE).contiguous()\n        self.destination=model.graph.destination.detach().to(DEVICE).contiguous(); self.edge=model.graph.edge_logits.weight.detach().to(DEVICE,dtype=AMP_DTYPE).contiguous(); concepts=torch.arange(model.n_concepts,device=DEVICE);gs=torch.exp(model.graph.scale.detach()); self.dest_key=(model.space.key(concepts).detach().float()*gs).to(AMP_DTYPE).contiguous(); self.concept_value=model.space.value(concepts).detach().to(AMP_DTYPE).contiguous(); self.message_weight=model.message_proj.weight.detach().to(AMP_DTYPE).contiguous(); self.norm_weight=model.norm.weight.detach().to(AMP_DTYPE).contiguous(); self.norm_bias=model.norm.bias.detach().to(AMP_DTYPE).contiguous()\n        self.fresh_ids=torch.empty(4,dtype=torch.int32,device=DEVICE);self.fresh_mass=torch.empty(4,dtype=torch.float32,device=DEVICE);self.query=torch.empty(16,dtype=torch.float32,device=DEVICE);self.next_ids=torch.empty(8,dtype=torch.int32,device=DEVICE);self.next_mass=torch.empty(8,dtype=torch.float32,device=DEVICE);self.hidden=torch.empty(64,dtype=torch.float32,device=DEVICE)\n\nclass FastWalkerRetriever:\n    def __init__(self,model,support,degree):\n        self.a=FastWalkerArtifacts(model);self.degree=int(degree);self.candidates=8*self.degree;self.blocks=self.candidates//128;assert self.blocks*128==self.candidates\n        self.support=support["item"][:,:degree].to(DEVICE,dtype=torch.int32).contiguous();self.bi=torch.empty(self.blocks*CFG.block_keep,dtype=torch.int32,device=DEVICE);self.bs=torch.empty(self.blocks*CFG.block_keep,dtype=torch.float32,device=DEVICE);self.out=torch.empty(10,dtype=torch.int32,device=DEVICE);self.merge_block=triton.next_power_of_2(self.blocks*CFG.block_keep)\n    @torch.inference_mode()\n    def step(self,item_id,state_ids,state_mass):\n        a=self.a;route_kernel[(1,)](item_id,a.item,a.query_weight,a.left_keys,a.right_keys,a.fresh_ids,a.fresh_mass,a.query,D=64,H=16,SIDE=256,INPUT_SCALE=8.,num_warps=4);walk_kernel[(1,)](state_ids,state_mass,a.fresh_ids,a.fresh_mass,a.query,a.destination,a.edge,a.dest_key,a.next_ids,a.next_mass,K=8,DEGREE=4,H=16,num_warps=4);readout_kernel[(1,)](item_id,a.next_ids,a.next_mass,a.concept_value,a.message_weight,a.norm_weight,a.norm_bias,a.item,a.hidden,K=8,D=64,INPUT_SCALE=8.,num_warps=4);terminal_block_kernel[(self.blocks,)](a.hidden,a.next_ids,self.support,a.item,self.bi,self.bs,DEGREE=self.degree,D=64,TOTAL_CANDIDATES=self.candidates,KEEP=CFG.block_keep,num_warps=4);terminal_merge_kernel[(1,)](self.bi,self.bs,self.out,COUNT=self.blocks*CFG.block_keep,BLOCK=self.merge_block,num_warps=4);return self.out\n\n# %%\n# ============================================================\n# 20. ONE DATASET × SEED RUN\n# ============================================================\n\ndef run_dataset_seed(dataset_name,seed,max_len_override=None,model_subset=None):\n    seed_all(seed); data,stats=load_dataset(dataset_name); train,train_ts,val_prefix,val_target,test_prefix,test_target=split_data(data); n_items=data["num_items"]\n    max_len=max_len_override or CFG.max_len_override or DATASETS[dataset_name].default_max_len\n    outdir=ROOT/dataset_name/f"L{max_len}";outdir.mkdir(parents=True,exist_ok=True);json.dump(stats,open(outdir/"dataset_stats.json","w"),indent=2)\n    ds=WindowDataset(train,max_len,seed); models=model_subset or CFG.models; rows=[]; terminal_rows=[]; latency_rows=[]\n    for name in models:\n        if name=="MostPop":\n            model=MostPopModel(n_items,data["frequency"]).to(DEVICE); info={"best_epoch":0,"train_seconds":0,"loss_mode":"none"}\n        else:\n            if name=="Mamba4Rec" and not HAS_MAMBA:\n                rows.append({"dataset":dataset_name,"seed":seed,"model":name,"status":"SKIPPED_MAMBA_NOT_INSTALLED"});continue\n            model=build_model(name,n_items,max_len,dataset_name).to(DEVICE);print("\\n",dataset_name,seed,name,"params",n_params(model))\n            if name=="SparseWalker-E2E":\n                pop=sorted(data["frequency"],key=lambda i:data["frequency"][i],reverse=True)\n                model.initialize_terminal(pop)\n            info=train_model(name,model,ds,val_prefix,val_target,n_items,max_len,data["frequency"],outdir,seed,DATASETS[dataset_name].large_catalog,dataset_name)\n        full=evaluate_full(model,test_prefix,test_target,n_items,max_len,data["frequency"],save_top=CFG.save_predictions); sampled=evaluate_sampled(model,test_prefix,test_target,n_items,max_len,CFG.sampled_eval_negs)\n        row={"dataset":dataset_name,"seed":seed,"model":name,"status":"OK","max_len":max_len,"params":n_params(model),"train_seconds":info.get("train_seconds",0),"best_epoch":info.get("best_epoch",0),"loss_mode":info.get("loss_mode","none"),**{k:v for k,v in full.items() if k!="top_items"},**sampled};rows.append(row);print("TEST",row)\n        if CFG.run_latency and torch.cuda.is_available():\n            t=benchmark_dense_request(model,test_prefix,max_len);latency_rows.append({"dataset":dataset_name,"seed":seed,"model":name,"kind":"full_request_full_catalog","max_len":max_len,**t})\n        if name=="SparseWalker":\n            support=compile_terminal_support(model,train,data["frequency"],n_items,max_len,outdir,seed); vf=final_walker_features(model,val_prefix,val_target,max_len);tf=final_walker_features(model,test_prefix,test_target,max_len)\n            for degree in CFG.degree_sweep:\n                vr=evaluate_terminal(model,support,vf,val_prefix,n_items,degree,data["frequency"]);tr=evaluate_terminal(model,support,tf,test_prefix,n_items,degree,data["frequency"]);rr={"dataset":dataset_name,"seed":seed,**tr,"val_NDCG@10":vr["NDCG@10"],"dense_walker_NDCG@10":full["NDCG@10"],"quality_retained":tr["NDCG@10"]/max(1e-12,full["NDCG@10"]),"terminal_MB":model.n_concepts*degree*4/1024**2,"compiler_events":support["compiler_events"],"mean_real_edges":float(support["observed_counts"].float().mean()),"support_saturation":float((support["observed_counts"]>=degree).float().mean())}\n                if CFG.run_latency and torch.cuda.is_available() and HAS_TRITON and d_model_is_fusable(model):\n                    ret=FastWalkerRetriever(model,support,degree);item=torch.tensor([1],dtype=torch.int32,device=DEVICE);sid=torch.randint(0,model.n_concepts,(8,),dtype=torch.int32,device=DEVICE);sm=torch.full((8,),1/8,device=DEVICE);ret.step(item,sid,sm);torch.cuda.synchronize();tim=benchmark_cuda(lambda:ret.step(item,sid,sm));rr.update({"terminal_median_ms":tim["median_ms"],"terminal_p95_ms":tim["p95_ms"]})\n                rr["terminal_source"]="posthoc_teacher"\n                terminal_rows.append(rr);print("TERMINAL",rr)\n        elif name=="SparseWalker-E2E":\n            support={"item":model.terminal_items.detach().cpu(),"compiler_events":0,\n                     "observed_counts":torch.zeros(model.n_concepts,dtype=torch.long)}\n            vf=final_walker_features(model,val_prefix,val_target,max_len)\n            tf=final_walker_features(model,test_prefix,test_target,max_len)\n            for degree in [d for d in CFG.degree_sweep if d<=model.terminal_degree]:\n                vr=evaluate_terminal(model,support,vf,val_prefix,n_items,degree,data["frequency"])\n                tr=evaluate_terminal(model,support,tf,test_prefix,n_items,degree,data["frequency"])\n                rr={"dataset":dataset_name,"seed":seed,**tr,\n                    "val_NDCG@10":vr["NDCG@10"],\n                    "dense_walker_NDCG@10":float("nan"),\n                    "quality_retained":float("nan"),\n                    "terminal_MB":model.n_concepts*degree*4/1024**2,\n                    "compiler_events":0,\n                    "mean_real_edges":float("nan"),\n                    "support_saturation":float("nan"),\n                    "terminal_source":"e2e_supervised"}\n                if CFG.run_latency and torch.cuda.is_available() and HAS_TRITON and d_model_is_fusable(model):\n                    ret=FastWalkerRetriever(model,support,degree)\n                    item=torch.tensor([1],dtype=torch.int32,device=DEVICE)\n                    sid=torch.randint(0,model.n_concepts,(8,),dtype=torch.int32,device=DEVICE)\n                    sm=torch.full((8,),1/8,device=DEVICE)\n                    ret.step(item,sid,sm);torch.cuda.synchronize()\n                    tim=benchmark_cuda(lambda:ret.step(item,sid,sm))\n                    rr.update({"terminal_median_ms":tim["median_ms"],"terminal_p95_ms":tim["p95_ms"]})\n                terminal_rows.append(rr);print("TERMINAL-E2E",rr)\n        del model;gc.collect();\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n    rdf=pd.DataFrame(rows);tdf=pd.DataFrame(terminal_rows);ldf=pd.DataFrame(latency_rows)\n    rdf.to_csv(outdir/f"results_seed{seed}.csv",index=False)\n    if len(tdf):tdf.to_csv(outdir/f"terminal_seed{seed}.csv",index=False)\n    if len(ldf):ldf.to_csv(outdir/f"latency_seed{seed}.csv",index=False)\n    return rdf,tdf,ldf,stats\n\n\ndef d_model_is_fusable(model):\n    return isinstance(model,(SparseWalkerModel,SparseWalkerE2EModel)) and model.d_model==64 and model.h==16 and model.side==256 and model.active==8 and model.degree==4 and model.layers_n==2\n\n# %%\n# ============================================================\n# 21. AGGREGATION / PAPER TABLES\n# ============================================================\n\ndef mean_std_table(df,metrics=("HR@10","NDCG@10","MRR@10","Coverage@10","TailHR@10","sampled_NDCG@10")):\n    ok=df[df.get("status","OK")=="OK"].copy() if "status" in df else df.copy(); cols=[m for m in metrics if m in ok.columns]\n    g=ok.groupby(["dataset","model"])[cols].agg(["mean","std"]);return g.reset_index()\n\n\ndef paper_summary_flat(df, metrics=("HR@10","NDCG@10","MRR@10","Coverage@10","TailHR@10","sampled_NDCG@10")):\n    ok = df[df["status"] == "OK"].copy() if "status" in df.columns else df.copy()\n    rows=[]\n    for (dataset,model),g in ok.groupby(["dataset","model"]):\n        r={"dataset":dataset,"model":model,"seeds":int(g["seed"].nunique()) if "seed" in g else len(g)}\n        for m in metrics:\n            if m not in g.columns: continue\n            v=pd.to_numeric(g[m],errors="coerce").dropna()\n            if not len(v): continue\n            mean=float(v.mean()); std=float(v.std(ddof=1)) if len(v)>1 else 0.0; ci=1.96*std/math.sqrt(len(v)) if len(v)>1 else 0.0\n            r[f"{m}_mean"]=mean; r[f"{m}_std"]=std; r[f"{m}_ci95"]=ci\n        rows.append(r)\n    return pd.DataFrame(rows)\n\n\ndef add_relative_to_sas(df):\n    x=df.copy()\n    if "NDCG@10" not in x:return x\n    base=x[x.model=="SASRec"][["dataset","seed","NDCG@10"]].rename(columns={"NDCG@10":"sas_NDCG"});x=x.merge(base,on=["dataset","seed"],how="left");x["NDCG_rel_vs_SAS"]=(x["NDCG@10"]/x["sas_NDCG"]-1)*100;return x\n\n# %%\n# ============================================================\n# 22. OPTIONAL LONG-CONTEXT QUALITY ABLATION\n# ============================================================\n\ndef run_long_context_ablation():\n    if not CFG.run_long_context_ablation:return pd.DataFrame()\n    allr=[]\n    for L in CFG.long_context_lengths:\n        r,_,_,_=run_dataset_seed("ml20m",CFG.long_context_seed,max_len_override=L,model_subset=CFG.long_context_models);allr.append(r)\n    out=pd.concat(allr,ignore_index=True);out.to_csv(ROOT/"long_context_quality.csv",index=False);return out\n\n# %%\n# ============================================================\n# 23. MAIN\n# ============================================================\n\ndef run_all():\n    all_rows=[];all_terminal=[];all_latency=[];stats=[]\n    for d in CFG.datasets:\n        for seed in CFG.seeds:\n            r,t,l,s=run_dataset_seed(d,seed);all_rows.append(r);stats.append(s)\n            if len(t):all_terminal.append(t)\n            if len(l):all_latency.append(l)\n    results=pd.concat(all_rows,ignore_index=True);results=add_relative_to_sas(results);results.to_csv(ROOT/"all_results.csv",index=False)\n    summary=mean_std_table(results);summary.to_csv(ROOT/"summary_mean_std.csv",index=False)\n    paper_summary_flat(results).to_csv(ROOT/"paper_summary_flat.csv",index=False)\n    if all_terminal:\n        term=pd.concat(all_terminal,ignore_index=True);term.to_csv(ROOT/"all_terminal.csv",index=False)\n        term_summary=term.groupby([c for c in ["dataset","terminal_source","degree"] if c in term.columns])[[c for c in ["NDCG@10","HR@10","quality_retained","candidate_recall","teacher_top10_recall","terminal_median_ms","terminal_MB"] if c in term.columns]].agg(["mean","std"]).reset_index();term_summary.to_csv(ROOT/"terminal_summary.csv",index=False)\n    else:term=pd.DataFrame();term_summary=pd.DataFrame()\n    if all_latency:\n        lat=pd.concat(all_latency,ignore_index=True);lat.to_csv(ROOT/"all_latency.csv",index=False)\n    else:lat=pd.DataFrame()\n    pd.DataFrame(stats).drop_duplicates("dataset").to_csv(ROOT/"dataset_stats.csv",index=False)\n    long_df=run_long_context_ablation()\n    with open(ROOT/"config.json","w") as f: json.dump(asdict(CFG),f,indent=2)\n    # zip paper artifacts (CSVs/config only; checkpoints remain in folders)\n    zpath=ROOT/"paper_tables.zip"\n    with zipfile.ZipFile(zpath,"w",zipfile.ZIP_DEFLATED) as z:\n        for p in ROOT.glob("*.csv"):z.write(p,p.name)\n        z.write(ROOT/"config.json","config.json")\n    print("\\n=== MEAN ± STD ===");print(summary.to_string(index=False));print("\\nSaved:",ROOT)\n    return results,summary,term,term_summary,lat,long_df\n\nif __name__ == "__main__":\n    RESULTS, SUMMARY, TERMINAL, TERMINAL_SUMMARY, LATENCY, LONG_CONTEXT = run_all()\n'
Path('/content/flyrec_benchmark.py').write_text(BENCHMARK_SOURCE)
import flyrec_benchmark as b


In [ ]:
# Pilot: one seed. For a serious comparison, use >=3 seeds and multiple subgraphs.
DATASET = 'beauty'  # also 'ml1m'
SEEDS = (42,)
MAX_NODES = 512
CHANNELS = 8
GRAPH_SEED = 2026
MAX_EPOCHS = 30
CONNECTOME_RELEASE = 'SET_THIS_TO_THE_DOWNLOADED_RELEASE'
CONNECTOME_PATH = ''  # Set a local path, or leave empty for upload.
if not CONNECTOME_PATH:
    from google.colab import files
    uploaded = files.upload()
    candidates = [n for n in uploaded if n.endswith(('.csv', '.csv.gz'))]
    assert len(candidates)==1, 'Upload exactly one connections CSV or CSV.gz.'
    CONNECTOME_PATH = candidates[0]
assert CONNECTOME_RELEASE != 'SET_THIS_TO_THE_DOWNLOADED_RELEASE', 'Set CONNECTOME_RELEASE before continuing.'


In [ ]:
import csv, gzip, random
from collections import Counter

def read_edges(path, max_nodes):
    opener = gzip.open if str(path).endswith('.gz') else open
    def rows():
        with opener(path, 'rt', newline='') as f:
            reader = csv.DictReader(f)
            required={'pre_root_id','post_root_id'}
            if not required.issubset(reader.fieldnames or []):
                raise ValueError('Required columns: pre_root_id, post_root_id')
            for r in reader:
                # Strings preserve 64-bit neuron identifiers exactly.
                u,v=r['pre_root_id'].strip(),r['post_root_id'].strip()
                if not u or not v or u==v: continue
                if 'syn_count' in r and float(r['syn_count'])<=0: continue
                yield u,v
    # Deduplicate neuropil rows before calculating degrees.
    edges=set(rows())
    degree=Counter()
    for u,v in edges: degree[u]+=1; degree[v]+=1
    nodes=sorted(degree, key=lambda x:(-degree[x],x))[:max_nodes]
    if len(nodes)<8: raise ValueError('Connectome has too few nodes.')
    ids={v:i for i,v in enumerate(nodes)}
    selected=sorted((ids[u],ids[v]) for u,v in edges if u in ids and v in ids)
    if len(selected)<len(nodes): raise ValueError('Induced graph too sparse; choose another subset.')
    return nodes, selected

def rewire(edges, n, seed, swaps_per_edge=10):
    # Directed double-edge swaps preserve every node's in/out degree exactly.
    rng=random.Random(seed); out=list(edges); occupied=set(out)
    goal=swaps_per_edge*len(out); accepted=0; attempts=0
    while accepted<goal and attempts<goal*50:
        attempts+=1
        i,j=rng.sample(range(len(out)),2)
        a,c=out[i]; d,e=out[j]
        x,y=(a,e),(d,c)
        if a==d or c==e or a==e or d==c: continue
        if x in occupied or y in occupied: continue
        occupied.remove(out[i]); occupied.remove(out[j])
        occupied.add(x); occupied.add(y); out[i]=x; out[j]=y
        accepted+=1
    if accepted<goal: raise RuntimeError('Insufficient accepted rewiring swaps.')
    assert Counter(u for u,v in edges)==Counter(u for u,v in out)
    assert Counter(v for u,v in edges)==Counter(v for u,v in out)
    return sorted(out), accepted


In [ ]:
nodes, biological_edges = read_edges(CONNECTOME_PATH, MAX_NODES)
rewired_edges, accepted_swaps = rewire(biological_edges, len(nodes), GRAPH_SEED)
overlap=len(set(biological_edges)&set(rewired_edges))/len(biological_edges)
manifest=dict(release=CONNECTOME_RELEASE, nodes=len(nodes), edges=len(biological_edges),
    selection='top total degree induced subgraph; unweighted; no self edges',
    channels=CHANNELS, graph_seed=GRAPH_SEED, accepted_swaps=accepted_swaps,
    edge_overlap=overlap, source_sha256=hashlib.sha256(Path(CONNECTOME_PATH).read_bytes()).hexdigest(),
    benchmark_sha256=hashlib.sha256(BENCHMARK_SOURCE.encode()).hexdigest(),
    dataset=DATASET, seeds=SEEDS, status='pilot; not a whole-brain reproduction')
(RUN_DIR/'graph_manifest.json').write_text(json.dumps(manifest,indent=2))
(RUN_DIR/'graphs.json').write_text(json.dumps(dict(node_ids=nodes,biological=biological_edges,rewired=rewired_edges)))
print(json.dumps(manifest,indent=2))


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FlyRec(b.ARRecommender):
    def __init__(self,n_items,max_len,edges,n_nodes,d=64,channels=8,dropout=.2):
        super().__init__(n_items,max_len,d)
        self.n_nodes=n_nodes; self.channels=channels
        self.item=nn.Embedding(n_items+2,d,padding_idx=0); b.init_embedding(self.item)
        # Receiver-row normalization; edge direction is source -> destination.
        idx=torch.tensor(edges,dtype=torch.long).T
        degree=torch.bincount(idx[1],minlength=n_nodes).float().clamp_min(1)
        adj=torch.sparse_coo_tensor(torch.stack([idx[1],idx[0]]),1/degree[idx[1]],(n_nodes,n_nodes)).coalesce()
        self.register_buffer('adj',adj)
        self.input=nn.Linear(d,channels)
        self.input_gain=nn.Parameter(torch.randn(n_nodes,channels)*.1)
        self.descriptor=nn.Parameter(torch.randn(n_nodes,channels)*.02)
        self.update=nn.GRUCell(channels*2,channels)
        self.pool=nn.Parameter(torch.randn(n_nodes,8)*.1)
        self.output=nn.Linear(channels*8,d)
        self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)

    @property
    def item_weight(self): return self.item.weight

    def step(self,item_ids,state):
        # Use float32 for sparse recurrence; fixed adjacency, learned dynamics.
        with torch.autocast(device_type=item_ids.device.type,enabled=False):
            x=self.input(self.drop(self.item(item_ids)).float())
            injected=x[:,None,:]*self.input_gain[None,:,:]
            pre=state+injected
            flat=pre.permute(1,0,2).reshape(self.n_nodes,-1)
            message=torch.sparse.mm(self.adj,flat).reshape(self.n_nodes,-1,self.channels).permute(1,0,2)
            desc=self.descriptor[None].expand(state.shape[0],-1,-1)
            inp=torch.cat([message+injected,desc],dim=-1)
            candidate=self.update(inp.reshape(-1,2*self.channels),state.reshape(-1,self.channels)).reshape_as(state)
            state=torch.where((item_ids!=0)[:,None,None],candidate,state)
            pooled=torch.einsum('bnc,np->bpc',state,self.pool.softmax(dim=0))
            h=self.norm(self.output(pooled.flatten(1)))
            return h,state

    def encode(self,seq):
        state=torch.zeros(seq.shape[0],self.n_nodes,self.channels,device=seq.device)
        outputs=[]
        for t in range(seq.shape[1]):
            h,state=self.step(seq[:,t],state); outputs.append(h)
        return torch.stack(outputs,dim=1)


The next cell checks causal behavior, padding, streaming parity, graph sensitivity and
backpropagation on a tiny synthetic fixture. Passing it verifies mechanics only.
It is **not recommendation-quality evidence** and the fixture never enters the benchmark.


In [ ]:
def check_model():
    torch.manual_seed(7)
    fixture=[(i,(i+1)%12) for i in range(12)]+[(i,(i+4)%12) for i in range(12)]
    m=FlyRec(30,8,fixture,12,d=16,channels=4,dropout=0).to(b.DEVICE).eval()
    x=torch.tensor([[1,2,3,4],[5,6,7,8]],device=b.DEVICE)
    h=m.encode(x)
    torch.testing.assert_close(h[:,:2],m.encode(x[:,:2]),atol=1e-5,rtol=1e-5)
    padded=m.encode(F.pad(x,(0,2)))
    torch.testing.assert_close(h,padded[:,:4],atol=1e-5,rtol=1e-5)
    torch.testing.assert_close(padded[:,4],h[:,-1],atol=1e-5,rtol=1e-5)
    state=torch.zeros(2,12,4,device=b.DEVICE)
    for t in range(4):
        z,state=m.step(x[:,t],state)
        torch.testing.assert_close(z,h[:,t],atol=1e-5,rtol=1e-5)
    saved=m.adj
    m.adj=torch.sparse_coo_tensor(saved.indices(),torch.zeros_like(saved.values()),saved.shape).coalesce()
    assert (m.encode(x)-h).abs().max()>1e-6, 'Graph has no measurable influence.'
    m.adj=saved
    opt=torch.optim.Adam(m.parameters(),lr=.001)
    loss=F.cross_entropy(m.score_hidden(m.encode(x)[:,-1]),torch.tensor([9,10],device=b.DEVICE))
    loss.backward()
    assert torch.isfinite(loss)
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in m.parameters())
    assert m.input_gain.grad.abs().sum()>0
    opt.step()
    print('Mechanics checks passed; no quality result implied.')
check_model()


## Controlled pilot

SASRec, GRU4Rec, FlyRec-biological and FlyRec-rewired share the dataset, splits,
embedding width, full cross-entropy objective, batch size, learning rate and training budget.
The biological/rewired pair has exactly equal trainable parameter counts and initialization.
SASRec and GRU parameter counts are reported, not artificially called matched.
No hyperparameter search is included; equal settings are not equally optimal settings.
Beauty is the default. ML1M also uses full CE here to keep the objective identical.
Validation selects checkpoints; test data is used only for final evaluation.

Read full-catalog NDCG@10 and HR@10, elapsed training time and parameters. The optional
latency output measures a complete history replay plus full-catalog scoring; it does not
compare cached streaming performance. Subgraph choice and one seed limit conclusions.


In [ ]:
b.CFG.datasets=(DATASET,); b.CFG.seeds=SEEDS
b.CFG.models=('SASRec','GRU4Rec','FlyRec-biological','FlyRec-rewired')
b.CFG.only_model=None; b.CFG.only_dataset=None; b.CFG.only_seed=None
b.CFG.batch_size=32; b.CFG.eval_batch_size=32
b.CFG.max_epochs=MAX_EPOCHS; b.CFG.patience=6; b.CFG.reuse=False
b.CFG.topks=(10,20); b.CFG.run_latency=False
b.CFG.max_len_override=50
b.CFG.lr_by_model.update({name:1e-3 for name in b.CFG.models})
original_factory=b.build_model
original_run_dataset_seed=b.run_dataset_seed
ACTIVE_SEED=42
def seeded_run(dataset_name,seed,*args,**kwargs):
    global ACTIVE_SEED
    ACTIVE_SEED=seed
    return original_run_dataset_seed(dataset_name,seed,*args,**kwargs)
b.run_dataset_seed=seeded_run
def factory(name,n_items,max_len,dataset_name=None):
    b.seed_all(ACTIVE_SEED)
    if name.startswith('FlyRec-'):
        edges=biological_edges if name=='FlyRec-biological' else rewired_edges
        return FlyRec(n_items,max_len,edges,len(nodes),d=b.CFG.d_model,channels=CHANNELS,dropout=b.CFG.dropout)
    return original_factory(name,n_items,max_len,dataset_name)
b.build_model=factory
b.loss_mode_for=lambda *args,**kwargs:'full'
(RUN_DIR/'experiment_config.json').write_text(json.dumps(dict(
    dataset=DATASET,seeds=SEEDS,models=b.CFG.models,batch_size=b.CFG.batch_size,
    max_epochs=MAX_EPOCHS,max_len=50,objective='full CE',learning_rate=.001,
    parameter_matching='biological vs rewired only'),indent=2))
RESULTS,SUMMARY,_,_,_,_=b.run_all()
display(RESULTS[['dataset','seed','model','params','NDCG@10','HR@10','train_seconds']])
print('Results:',RUN_DIR)


## Interpretation and next experiment

- Biological > rewired across repeated seeds/subgraphs: evidence that this retained wiring helps this model.
- Both > SASRec: promising alternative encoder; tune baselines and match parameter/compute budgets next.
- Biological ≈ rewired: generic recurrence/sparsity may explain the benefit.
- Both lose: this pilot fails; it does not refute a full-connectome model or other input/readout mappings.

The 512-node pilot deliberately trades biological fidelity for tractable experimentation.
Do not extrapolate to a whole-brain result. Before a paper-quality claim, test multiple
graph selections, a no-message control, equal tuning budgets, >=3 training seeds, another
dataset and a quality/latency/memory comparison. Training cost may dominate SASRec because
this implementation loops over events. Cached state is fixed-size but is not automatically faster.


In [ ]:
import shutil
archive=shutil.make_archive(str(RUN_DIR)+'_results','zip',RUN_DIR)
from google.colab import files
files.download(archive)
